# 001 OSM Network Feasibility Validation Plan

神奈川県の道路ネットワーク一括取得に向けた技術検証ノートブック。

- 対象都道府県: 神奈川県
- 想定業務: ラストマイル配送
- 取得対象ネットワーク: `drive`
- 対応する計画書: `docs/brainstorming/001_osm_network_feasibility_validation_plan.md`


## 出力方針

- OSM から取得した生に近い道路ネットワーク: `data/raw/road_network/`
- 後処理済みネットワーク: `data/processed/`
- 検証メトリクス: `outputs/metrics/`
- 可視化結果: `outputs/figures/`

このノートブックでは、まず `data/raw/road_network/` に神奈川県一括取得結果を保存する前提で進める。


In [6]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'road_network'
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics'
OUTPUT_FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'

for path in [DATA_RAW_DIR, DATA_PROCESSED_DIR, OUTPUT_METRICS_DIR, OUTPUT_FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATA_RAW_DIR, OUTPUT_METRICS_DIR, OUTPUT_FIGURES_DIR


(WindowsPath('C:/Users/LCCadmin/OneDrive - 株式会社\u3000ロジクロス・コミュニケーション/ドキュメント/GitHub/vrp-road-network-optimization/data/raw/road_network'),
 WindowsPath('C:/Users/LCCadmin/OneDrive - 株式会社\u3000ロジクロス・コミュニケーション/ドキュメント/GitHub/vrp-road-network-optimization/outputs/metrics'),
 WindowsPath('C:/Users/LCCadmin/OneDrive - 株式会社\u3000ロジクロス・コミュニケーション/ドキュメント/GitHub/vrp-road-network-optimization/outputs/figures'))

In [7]:
PREFECTURE_NAME = 'Kanagawa, Japan'
NETWORK_TYPE = 'drive'
GRAPH_FILENAME = 'kanagawa_drive.graphml'
METRICS_FILENAME = 'kanagawa_drive_network_metrics.csv'

GRAPH_PATH = DATA_RAW_DIR / GRAPH_FILENAME
METRICS_PATH = OUTPUT_METRICS_DIR / METRICS_FILENAME

print(f'Prefecture: {PREFECTURE_NAME}')
print(f'Network type: {NETWORK_TYPE}')
print(f'Graph path: {GRAPH_PATH}')
print(f'Metrics path: {METRICS_PATH}')


Prefecture: Kanagawa, Japan
Network type: drive
Graph path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\raw\road_network\kanagawa_drive.graphml
Metrics path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_drive_network_metrics.csv


## 次の実装セルで行うこと

1. `osmnx` で神奈川県の `drive` ネットワークを取得する
2. ノード数、エッジ数、処理時間、ファイルサイズを記録する
3. `GraphML` で保存し、再読込できることを確認する


In [5]:
import time

import osmnx as ox
import pandas as pd


In [ ]:
start_time = time.perf_counter()

graph = ox.graph_from_place(PREFECTURE_NAME, network_type=NETWORK_TYPE)

elapsed_seconds = time.perf_counter() - start_time

ox.save_graphml(graph, GRAPH_PATH)
reloaded_graph = ox.load_graphml(GRAPH_PATH)

file_size_mb = GRAPH_PATH.stat().st_size / (1024 * 1024)
metrics_df = pd.DataFrame(
    [
        {
            'prefecture_name': PREFECTURE_NAME,
            'network_type': NETWORK_TYPE,
            'node_count': graph.number_of_nodes(),
            'edge_count': graph.number_of_edges(),
            'elapsed_seconds': round(elapsed_seconds, 2),
            'graph_path': str(GRAPH_PATH),
            'file_size_mb': round(file_size_mb, 2),
            'reload_node_count': reloaded_graph.number_of_nodes(),
            'reload_edge_count': reloaded_graph.number_of_edges(),
        }
    ]
)

metrics_df.to_csv(METRICS_PATH, index=False, encoding='utf-8-sig')
metrics_df


In [ ]:
print(f'Download completed: {GRAPH_PATH}')
print(f'Metrics saved: {METRICS_PATH}')
print(f'Graph file size (MB): {file_size_mb:.2f}')
print(f'Nodes: {graph.number_of_nodes()}')
print(f'Edges: {graph.number_of_edges()}')
print(f'Reload nodes: {reloaded_graph.number_of_nodes()}')
print(f'Reload edges: {reloaded_graph.number_of_edges()}')
print(f'Elapsed seconds: {elapsed_seconds:.2f}')


Download completed: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\raw\road_network\kanagawa_drive.graphml
Metrics saved: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_drive_network_metrics.csv
Graph file size (MB): 259.52
Nodes: 223850
Edges: 608205
Reload nodes: 223850
Reload edges: 608205
Elapsed seconds: 296.38


## Step 3. 県全体ネットワークの後処理

このセルでは、Step 2 で取得した生の `GraphML` を、後続の距離計算や VRP 前処理に使いやすい形へ整える。

実施内容:
- 生ネットワークから最大弱連結成分のみを残す
- 車配送の routing に必要な属性を残し、それ以外を削減する
- `maxspeed` をもとに `travel_time` を付与する
- 後処理済み `GraphML` を `data/processed/kanagawa_drive_step3/` に保存する
- 保存後に再読込し、ノード・エッジ・属性差分を確認する
- 実行ログを `logs/` に保存し、メトリクスを表示する


In [8]:
from datetime import datetime
from pathlib import Path
import math

import networkx as nx

STEP3_OUTPUT_DIR = DATA_PROCESSED_DIR / 'kanagawa_drive_step3'
STEP3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = PROJECT_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

processed_graph_path = STEP3_OUTPUT_DIR / 'kanagawa_drive_processed.graphml'
processed_metrics_path = STEP3_OUTPUT_DIR / 'kanagawa_drive_processed_metrics.csv'
timestamp_label = datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = LOG_DIR / f'001_osm_network_feasibility_validation_plan_step3_{timestamp_label}.log'

keep_graph_attrs = {'created_date', 'created_with', 'crs', 'simplified'}
keep_node_attrs = {'x', 'y', 'street_count', 'highway', 'junction', 'ref'}
keep_edge_attrs = {
    'osmid',
    'highway',
    'oneway',
    'reversed',
    'length',
    'access',
    'lanes',
    'maxspeed',
    'name',
    'ref',
    'geometry',
    'bridge',
    'tunnel',
    'junction',
    'travel_time',
}


def normalize_value(value):
    if hasattr(value, 'wkt'):
        return value.wkt
    if isinstance(value, float):
        if math.isnan(value):
            return 'NaN'
        return round(value, 6)
    if isinstance(value, (list, tuple, set)):
        return tuple(normalize_value(item) for item in value)
    if isinstance(value, dict):
        return tuple(sorted((key, normalize_value(val)) for key, val in value.items()))
    return value


def prune_graph_attributes(graph):
    graph.graph = {key: value for key, value in graph.graph.items() if key in keep_graph_attrs}
    for _, data in graph.nodes(data=True):
        remove_keys = [key for key in data if key not in keep_node_attrs]
        for key in remove_keys:
            del data[key]
    for _, _, _, data in graph.edges(keys=True, data=True):
        remove_keys = [key for key in data if key not in keep_edge_attrs]
        for key in remove_keys:
            del data[key]


def compare_graphs(left_graph, right_graph, max_differences=20):
    differences = []

    left_graph_attrs = {key: normalize_value(value) for key, value in left_graph.graph.items()}
    right_graph_attrs = {key: normalize_value(value) for key, value in right_graph.graph.items()}
    if left_graph_attrs != right_graph_attrs:
        differences.append(
            f'Graph attributes differ: left={left_graph_attrs}, right={right_graph_attrs}'
        )

    left_nodes = set(left_graph.nodes)
    right_nodes = set(right_graph.nodes)
    only_left_nodes = sorted(left_nodes - right_nodes)
    only_right_nodes = sorted(right_nodes - left_nodes)
    for node_id in only_left_nodes[:max_differences]:
        differences.append(f'Node only in processed graph: {node_id}')
    for node_id in only_right_nodes[:max_differences]:
        differences.append(f'Node only in reloaded graph: {node_id}')

    shared_nodes = sorted(left_nodes & right_nodes)
    for node_id in shared_nodes:
        left_attrs = {key: normalize_value(value) for key, value in left_graph.nodes[node_id].items()}
        right_attrs = {key: normalize_value(value) for key, value in right_graph.nodes[node_id].items()}
        if left_attrs != right_attrs:
            differences.append(
                f'Node attribute difference at {node_id}: left={left_attrs}, right={right_attrs}'
            )
        if len(differences) >= max_differences:
            return differences[:max_differences]

    left_edges = set(left_graph.edges(keys=True))
    right_edges = set(right_graph.edges(keys=True))
    only_left_edges = sorted(left_edges - right_edges)
    only_right_edges = sorted(right_edges - left_edges)
    for edge_key in only_left_edges[:max_differences]:
        differences.append(f'Edge only in processed graph: {edge_key}')
    for edge_key in only_right_edges[:max_differences]:
        differences.append(f'Edge only in reloaded graph: {edge_key}')

    shared_edges = sorted(left_edges & right_edges)
    for edge_key in shared_edges:
        u, v, k = edge_key
        left_attrs = {
            key: normalize_value(value) for key, value in left_graph.get_edge_data(u, v, k).items()
        }
        right_attrs = {
            key: normalize_value(value) for key, value in right_graph.get_edge_data(u, v, k).items()
        }
        if left_attrs != right_attrs:
            differences.append(
                f'Edge attribute difference at {(u, v, k)}: left={left_attrs}, right={right_attrs}'
            )
        if len(differences) >= max_differences:
            return differences[:max_differences]

    return differences[:max_differences]


step3_start_time = time.perf_counter()
raw_graph = ox.load_graphml(GRAPH_PATH)
component_count_before = nx.number_weakly_connected_components(raw_graph)
largest_component_nodes = max(nx.weakly_connected_components(raw_graph), key=len)
processed_graph = raw_graph.subgraph(largest_component_nodes).copy()
removed_nodes = raw_graph.number_of_nodes() - processed_graph.number_of_nodes()
removed_edges = raw_graph.number_of_edges() - processed_graph.number_of_edges()

processed_graph = ox.add_edge_speeds(processed_graph)
processed_graph = ox.add_edge_travel_times(processed_graph)
prune_graph_attributes(processed_graph)

ox.save_graphml(processed_graph, processed_graph_path)
reloaded_processed_graph = ox.load_graphml(processed_graph_path)
step3_elapsed_seconds = time.perf_counter() - step3_start_time
processed_file_size_mb = processed_graph_path.stat().st_size / (1024 * 1024)
component_count_after = nx.number_weakly_connected_components(processed_graph)

differences = compare_graphs(processed_graph, reloaded_processed_graph)

metrics_step3_df = pd.DataFrame(
    [
        {
            'input_graph_path': str(GRAPH_PATH),
            'processed_graph_path': str(processed_graph_path),
            'raw_node_count': raw_graph.number_of_nodes(),
            'raw_edge_count': raw_graph.number_of_edges(),
            'processed_node_count': processed_graph.number_of_nodes(),
            'processed_edge_count': processed_graph.number_of_edges(),
            'removed_nodes': removed_nodes,
            'removed_edges': removed_edges,
            'component_count_before': component_count_before,
            'component_count_after': component_count_after,
            'processed_file_size_mb': round(processed_file_size_mb, 2),
            'elapsed_seconds': round(step3_elapsed_seconds, 2),
            'difference_count': len(differences),
        }
    ]
)
metrics_step3_df.to_csv(processed_metrics_path, index=False, encoding='utf-8-sig')

retained_node_attributes = sorted({key for _, data in processed_graph.nodes(data=True) for key in data.keys()})
retained_edge_attributes = sorted(
    {key for _, _, _, data in processed_graph.edges(keys=True, data=True) for key in data.keys()}
)

log_lines = [
    'Step 3 post-processing log',
    f'Created at: {datetime.now().isoformat()}',
    f'Input graph: {GRAPH_PATH}',
    f'Processed graph: {processed_graph_path}',
    f'Processed metrics: {processed_metrics_path}',
    f'Raw nodes: {raw_graph.number_of_nodes()}',
    f'Raw edges: {raw_graph.number_of_edges()}',
    f'Processed nodes: {processed_graph.number_of_nodes()}',
    f'Processed edges: {processed_graph.number_of_edges()}',
    f'Removed nodes: {removed_nodes}',
    f'Removed edges: {removed_edges}',
    f'Weakly connected components before: {component_count_before}',
    f'Weakly connected components after: {component_count_after}',
    f'Elapsed seconds: {step3_elapsed_seconds:.2f}',
    f'Processed file size (MB): {processed_file_size_mb:.2f}',
    f'Retained node attributes: {retained_node_attributes}',
    f'Retained edge attributes: {retained_edge_attributes}',
]

if differences:
    log_lines.append('Differences found between processed graph and reloaded graph:')
    log_lines.extend(differences)
else:
    log_lines.append('No differences found between processed graph and reloaded graph.')

log_path.write_text('\n'.join(log_lines) + '\n', encoding='utf-8')

print(f'Step 3 output directory: {STEP3_OUTPUT_DIR}')
print(f'Processed graph saved: {processed_graph_path}')
print(f'Processed metrics saved: {processed_metrics_path}')
print(f'Log saved: {log_path}')
print(f'Raw nodes: {raw_graph.number_of_nodes()}')
print(f'Raw edges: {raw_graph.number_of_edges()}')
print(f'Processed nodes: {processed_graph.number_of_nodes()}')
print(f'Processed edges: {processed_graph.number_of_edges()}')
print(f'Removed nodes: {removed_nodes}')
print(f'Removed edges: {removed_edges}')
print(f'Weakly connected components before: {component_count_before}')
print(f'Weakly connected components after: {component_count_after}')
print(f'Processed file size (MB): {processed_file_size_mb:.2f}')
print(f'Elapsed seconds: {step3_elapsed_seconds:.2f}')
if differences:
    print(f'Differences found after reload: {len(differences)}')
    for difference in differences:
        print(difference)
else:
    print('Differences found after reload: none')

metrics_step3_df


Step 3 output directory: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\processed\kanagawa_drive_step3
Processed graph saved: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\processed\kanagawa_drive_step3\kanagawa_drive_processed.graphml
Processed metrics saved: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\processed\kanagawa_drive_step3\kanagawa_drive_processed_metrics.csv
Log saved: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\logs\001_osm_network_feasibility_validation_plan_step3_20260317_170057.log
Raw nodes: 223850
Raw edges: 608205
Processed nodes: 223850
Processed edges: 608205
Removed nodes: 0
Removed edges: 0
Weakly connected components before: 1
Weakly connected components after: 1
Processed file size (MB): 285.75
Elapsed seconds: 160.85
Differences found after reload: none


,input_graph_path,processed_graph_path,raw_node_count,raw_edge_count,processed_node_count,processed_edge_count,removed_nodes,removed_edges,component_count_before,component_count_after,processed_file_size_mb,elapsed_seconds,difference_count
0,C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,223850,608205,223850,608205,0,0,1,1,285.75,160.85,0


# 市区町村単位の地図データ取得

神奈川県を市区町村単位に分割して道路ネットワークを順次取得し、Step 4 の結合検証に使うための生データを作成する。

## 実行内容

1. オープンデータを元に神奈川県の市区町村一覧を取得する
2. OSMnx に渡すための英語名へ整形する
3. 各市区町村について `drive` ネットワークを順次取得する
4. 取得した生データを `data/raw/road_network/kanagawa_municipalities/<municipality>.graphml` に保存する
5. 行政区ごとの取得成否、ノード数、エッジ数、ファイルサイズ、処理時間、エラー内容を記録する
6. 実行ログを `logs/` に保存する
7. 失敗した行政区があっても全体処理は継続し、失敗分だけ再実行できる形にする

## 想定入出力

- 入力: 神奈川県の市区町村一覧、対象都道府県 `Kanagawa, Japan`、ネットワーク種別 `drive`
- 出力: 市区町村単位の GraphML 群、取得メトリクス CSV、実行ログ

## 出力想定内容

- 市区町村単位で安定して分割取得できるか
- Step 4 の結合処理へ渡せる入力ファイル群を揃えられるか
- エラーが出た行政区だけを後から再実行できる運用にできるか

## 補足

- 今回保存するのは後処理前の生データとする
- 分割単位は市区町村単位とし、政令指定都市もまずは市単位で扱う
- 結果は Step 4 の分割データ結合検証へ引き継ぐ


In [9]:
import re

ADMIN_UNITS_PATH = PROJECT_ROOT / 'data' / 'raw' / 'admin_units' / 'kanagawa_municipalities.csv'
MUNICIPALITY_OUTPUT_DIR = DATA_RAW_DIR / 'kanagawa_municipalities'
MUNICIPALITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STEP3_METRICS_DIR = OUTPUT_METRICS_DIR / 'kanagawa_municipalities'
STEP3_METRICS_DIR.mkdir(parents=True, exist_ok=True)
STEP3_LOG_DIR = PROJECT_ROOT / 'logs'
STEP3_LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_LABEL = datetime.now().strftime('%Y%m%d_%H%M%S')
STEP3_LOG_PATH = STEP3_LOG_DIR / f'001_osm_network_feasibility_validation_plan_step3_fetch_{RUN_LABEL}.log'
STEP3_METRICS_PATH = STEP3_METRICS_DIR / f'kanagawa_municipality_fetch_metrics_{RUN_LABEL}.csv'
STEP3_METRICS_LATEST_PATH = STEP3_METRICS_DIR / 'kanagawa_municipality_fetch_metrics_latest.csv'
STEP3_FAILURES_PATH = STEP3_METRICS_DIR / f'kanagawa_municipality_failures_{RUN_LABEL}.csv'
STEP3_FAILURES_LATEST_PATH = STEP3_METRICS_DIR / 'kanagawa_municipality_failures_latest.csv'

# 必要に応じて再実行対象を指定する。空配列なら全件対象。
TARGET_MUNICIPALITIES = []
# 直近メトリクスで error になった行政区だけ再実行したい場合は True。
RERUN_ONLY_FAILED = False
# 既存の GraphML を上書きしたい場合だけ True。
OVERWRITE_EXISTING = False


def append_step3_log(message):
    with STEP3_LOG_PATH.open('a', encoding='utf-8') as log_file:
        log_file.write(message + '\n')


def slugify_filename(name):
    slug = re.sub(r'[^A-Za-z0-9]+', '_', str(name).strip().lower()).strip('_')
    return slug or 'municipality'


def flush_step3_metrics(records):
    metrics_df = pd.DataFrame(records)
    metrics_df.to_csv(STEP3_METRICS_PATH, index=False, encoding='utf-8-sig')
    metrics_df.to_csv(STEP3_METRICS_LATEST_PATH, index=False, encoding='utf-8-sig')
    failure_df = metrics_df[metrics_df['status'] == 'error'].copy()
    failure_df.to_csv(STEP3_FAILURES_PATH, index=False, encoding='utf-8-sig')
    failure_df.to_csv(STEP3_FAILURES_LATEST_PATH, index=False, encoding='utf-8-sig')
    return metrics_df, failure_df


def resolve_rerun_targets():
    if TARGET_MUNICIPALITIES:
        return set(TARGET_MUNICIPALITIES)
    if not RERUN_ONLY_FAILED:
        return None
    if not STEP3_METRICS_LATEST_PATH.exists():
        raise FileNotFoundError('RERUN_ONLY_FAILED=True ですが、直近メトリクスが見つかりません。')
    latest_df = pd.read_csv(STEP3_METRICS_LATEST_PATH)
    return set(latest_df.loc[latest_df['status'] == 'error', 'municipality_name_en'].dropna().tolist())


append_step3_log('Step 3 municipality fetch started')
append_step3_log(f'Created at: {datetime.now().isoformat()}')
append_step3_log(f'Admin units path: {ADMIN_UNITS_PATH}')
append_step3_log(f'Output directory: {MUNICIPALITY_OUTPUT_DIR}')
append_step3_log(f'Metrics path: {STEP3_METRICS_PATH}')
append_step3_log(f'Overwrite existing: {OVERWRITE_EXISTING}')
append_step3_log(f'Rerun only failed: {RERUN_ONLY_FAILED}')
append_step3_log(f'Target municipalities: {TARGET_MUNICIPALITIES}')

if not ADMIN_UNITS_PATH.exists():
    raise FileNotFoundError(f'行政区一覧ファイルが見つかりません: {ADMIN_UNITS_PATH}')

admin_df = pd.read_csv(ADMIN_UNITS_PATH, encoding='utf-8-sig')
admin_df = admin_df[admin_df['prefecture_name'].eq('Kanagawa')].copy()
target_set = resolve_rerun_targets()
if target_set is not None:
    admin_df = admin_df[admin_df['municipality_name_en'].isin(target_set)].copy()

admin_df = admin_df.sort_values(['priority', 'municipality_name_en'], ascending=[True, True]).reset_index(drop=True)
records = []

print(f'Admin units file: {ADMIN_UNITS_PATH}')
print(f'Target municipalities: {len(admin_df)}')
print(f'Output directory: {MUNICIPALITY_OUTPUT_DIR}')
print(f'Log path: {STEP3_LOG_PATH}')
print(f'Metrics path: {STEP3_METRICS_PATH}')

for index, row in enumerate(admin_df.itertuples(index=False), start=1):
    municipality_name_en = row.municipality_name_en
    municipality_name_ja = row.municipality_name_ja
    osm_query_name = row.osm_query_name
    municipality_type = row.municipality_type
    priority = row.priority
    output_filename = f"{slugify_filename(municipality_name_en)}.graphml"
    output_path = MUNICIPALITY_OUTPUT_DIR / output_filename
    started_at = datetime.now().isoformat()
    print(f'[{index}/{len(admin_df)}] {municipality_name_en} start')
    append_step3_log(f'[{index}/{len(admin_df)}] START {municipality_name_en} ({osm_query_name})')

    if output_path.exists() and not OVERWRITE_EXISTING:
        file_size_mb = round(output_path.stat().st_size / (1024 * 1024), 2)
        record = {
            'run_label': RUN_LABEL,
            'started_at': started_at,
            'municipality_name_ja': municipality_name_ja,
            'municipality_name_en': municipality_name_en,
            'municipality_type': municipality_type,
            'priority': priority,
            'osm_query_name': osm_query_name,
            'output_path': str(output_path),
            'status': 'skipped_existing',
            'node_count': pd.NA,
            'edge_count': pd.NA,
            'file_size_mb': file_size_mb,
            'elapsed_seconds': 0.0,
            'error_message': '',
        }
        records.append(record)
        flush_step3_metrics(records)
        print(f'[{index}/{len(admin_df)}] {municipality_name_en} skipped_existing ({file_size_mb} MB)')
        append_step3_log(f'[{index}/{len(admin_df)}] SKIP {municipality_name_en} existing_file={output_path}')
        continue

    started_perf = time.perf_counter()
    try:
        graph = ox.graph_from_place(osm_query_name, network_type=NETWORK_TYPE)
        ox.save_graphml(graph, output_path)
        elapsed_seconds = round(time.perf_counter() - started_perf, 2)
        file_size_mb = round(output_path.stat().st_size / (1024 * 1024), 2)
        record = {
            'run_label': RUN_LABEL,
            'started_at': started_at,
            'municipality_name_ja': municipality_name_ja,
            'municipality_name_en': municipality_name_en,
            'municipality_type': municipality_type,
            'priority': priority,
            'osm_query_name': osm_query_name,
            'output_path': str(output_path),
            'status': 'success',
            'node_count': graph.number_of_nodes(),
            'edge_count': graph.number_of_edges(),
            'file_size_mb': file_size_mb,
            'elapsed_seconds': elapsed_seconds,
            'error_message': '',
        }
        records.append(record)
        flush_step3_metrics(records)
        print(
            f"[{index}/{len(admin_df)}] {municipality_name_en} success "
            f"nodes={graph.number_of_nodes()} edges={graph.number_of_edges()} size_mb={file_size_mb} elapsed={elapsed_seconds}"
        )
        append_step3_log(
            f"[{index}/{len(admin_df)}] SUCCESS {municipality_name_en} nodes={graph.number_of_nodes()} "
            f"edges={graph.number_of_edges()} size_mb={file_size_mb} elapsed={elapsed_seconds}"
        )
    except Exception as exc:
        elapsed_seconds = round(time.perf_counter() - started_perf, 2)
        error_message = str(exc).replace('\n', ' ')[:1000]
        record = {
            'run_label': RUN_LABEL,
            'started_at': started_at,
            'municipality_name_ja': municipality_name_ja,
            'municipality_name_en': municipality_name_en,
            'municipality_type': municipality_type,
            'priority': priority,
            'osm_query_name': osm_query_name,
            'output_path': str(output_path),
            'status': 'error',
            'node_count': pd.NA,
            'edge_count': pd.NA,
            'file_size_mb': pd.NA,
            'elapsed_seconds': elapsed_seconds,
            'error_message': error_message,
        }
        records.append(record)
        flush_step3_metrics(records)
        print(f'[{index}/{len(admin_df)}] {municipality_name_en} error: {error_message}')
        append_step3_log(f'[{index}/{len(admin_df)}] ERROR {municipality_name_en}: {error_message}')

step3_fetch_metrics_df, step3_fetch_failure_df = flush_step3_metrics(records)

print('Step 3 municipality fetch completed')
print(f'Total records: {len(step3_fetch_metrics_df)}')
print(f"Success: {(step3_fetch_metrics_df['status'] == 'success').sum()}")
print(f"Skipped existing: {(step3_fetch_metrics_df['status'] == 'skipped_existing').sum()}")
print(f"Errors: {(step3_fetch_metrics_df['status'] == 'error').sum()}")
print(f'Failure list path: {STEP3_FAILURES_PATH}')

append_step3_log('Step 3 municipality fetch completed')
append_step3_log(f'Total records: {len(step3_fetch_metrics_df)}')
append_step3_log(f"Success: {(step3_fetch_metrics_df['status'] == 'success').sum()}")
append_step3_log(f"Skipped existing: {(step3_fetch_metrics_df['status'] == 'skipped_existing').sum()}")
append_step3_log(f"Errors: {(step3_fetch_metrics_df['status'] == 'error').sum()}")
append_step3_log(f'Failure list path: {STEP3_FAILURES_PATH}')

step3_fetch_metrics_df


Admin units file: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\raw\admin_units\kanagawa_municipalities.csv
Target municipalities: 33
Output directory: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\raw\road_network\kanagawa_municipalities
Log path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\logs\001_osm_network_feasibility_validation_plan_step3_fetch_20260318_111723.log
Metrics path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_municipalities\kanagawa_municipality_fetch_metrics_20260318_111723.csv
[1/33] Fujisawa start
[1/33] Fujisawa success nodes=11555 edges=32054 size_mb=12.97 elapsed=31.46
[2/33] Kawasaki start
[2/33] Kawasaki success nodes=19199 edges=52759 size_mb=23.04 elapsed=64.23
[3/33] Sagamihara start
[3/33] Sagamihara success nodes=16466 edges=46

,run_label,started_at,municipality_name_ja,municipality_name_en,municipality_type,priority,osm_query_name,output_path,status,node_count,edge_count,file_size_mb,elapsed_seconds,error_message
0,20260318_111723,2026-03-18T11:17:23.156677,藤沢市,Fujisawa,city,high,"Fujisawa, Kanagawa, Japan",C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,success,11555,32054,12.97,31.46,
1,20260318_111723,2026-03-18T11:17:54.700090,川崎市,Kawasaki,city,high,"Kawasaki, Kanagawa, Japan",C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,success,19199,52759,23.04,64.23,
2,20260318_111723,2026-03-18T11:18:59.056247,相模原市,Sagamihara,city,high,"Sagamihara, Kanagawa, Japan",C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,success,16466,46535,20.71,53.06,
3,20260318_111723,2026-03-18T11:19:52.206347,横浜市,Yokohama,city,high,"Yokohama, Kanagawa, Japan",C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,success,80163,219687,90.96,126.97,
4,20260318_111723,2026-03-18T11:21:59.607624,愛川町,Aikawa,town,low,"Aikawa, Kanagawa, Japan",C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,success,2588,6522,2.89,16.58,
5,20260318_111723,2026-03-18T11:22:16.204161,綾瀬市,Ayase,city,low,"Ayase, Kanagawa, Japan",C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,success,2733,7615,3.31,23.73,
6,20260318_111723,2026-03-18T11:22:39.955139,秦野市,Hadano,city,low,"Hadano, Kanagawa, Japan",C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,success,9298,22676,9.53,115.64,
7,20260318_111723,2026-03-18T11:24:35.641199,葉山町,Hayama,town,low,"Hayama, Kanagawa, Japan",C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,success,1344,3314,1.54,11.77,
8,20260318_111723,2026-03-18T11:24:47.420234,伊勢原市,Isehara,city,low,"Isehara, Kanagawa, Japan",C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,success,4681,12331,5.30,25.10,
9,20260318_111723,2026-03-18T11:25:12.543870,開成町,Kaisei,town,low,"Kaisei, Kanagawa, Japan",C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,success,624,1741,0.71,20.66,


# Step 4 実行前の依存関係確認

Step 4 の経路疎通確認では、実行環境によっては未投影グラフの最寄りノード探索で `scikit-learn` が必要になる場合がある。以下のセルでは `scikit-learn` の有無を確認する。なお、Step 4 のコード本体では投影済みグラフを使って最寄りノード探索するため、`scikit-learn` がなくても実行できるようにしている。


In [13]:
import importlib.util

if importlib.util.find_spec('sklearn') is None:
    print('scikit-learn is not installed in this Notebook environment. Step 4 will use a projected graph to avoid this dependency.')
else:
    import sklearn

    print(f'scikit-learn is already available: {sklearn.__version__}')


scikit-learn is already available: 1.8.0


# 分割取得データの結合と経路疎通確認

市区町村単位で取得した生データを 1 つのネットワークへ結合し、県全体として再利用できるかを確認する。

## 実行内容

1. 市区町村単位の GraphML を全件読み込む
2. ノード ID を基準に単純合成で 1 つのグラフへ結合する
3. 結合後グラフに `travel_time` を付与し、保存して再読込確認を行う
4. 結合結果を Step 2 の一括取得版および Step 2.5 の後処理版と比較する
5. ノード数、エッジ数、ファイルサイズ、処理時間を記録する
6. 有名地点を使って、結合後グラフで道のり距離が取得できるかを確認する
7. ログとメトリクスを保存する

## 想定入出力

- 入力: `data/raw/road_network/kanagawa_municipalities/*.graphml`
- 比較対象 1: `data/raw/road_network/kanagawa_drive.graphml`
- 比較対象 2: `data/processed/kanagawa_drive_step3/kanagawa_drive_processed.graphml`
- 出力: `data/processed/kanagawa_municipality_merged.graphml`、比較メトリクス CSV、経路疎通結果、実行ログ

## 出力想定内容

- 分割取得データを県全体グラフとして結合できるか
- 一括取得版と比べてノード数、エッジ数、サイズに大きな差がないか
- 結合後も有名地点間で道のり距離を取得できるか
- Step 5 以降の経路検証に進める状態か

## 補足

- 今回の結合方式は PoC 向けの単純合成とする
- Google Maps との比較は別 Step で実施する
- 主目的は、分割後に結合したネットワークでも道のり距離が途切れず取得できるかの確認である


In [14]:
from pyproj import Transformer

STEP4_INPUT_DIR = DATA_RAW_DIR / 'kanagawa_municipalities'
STEP4_OUTPUT_GRAPH_PATH = DATA_PROCESSED_DIR / 'kanagawa_municipality_merged.graphml'
STEP4_METRICS_PATH = OUTPUT_METRICS_DIR / 'kanagawa_municipality_merged_metrics.csv'
STEP4_ROUTE_PATH = OUTPUT_METRICS_DIR / 'kanagawa_municipality_merged_route_checks.csv'
STEP4_LOG_PATH = LOG_DIR / f"001_osm_network_feasibility_validation_plan_step4_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

STEP4_REFERENCE_RAW_PATH = GRAPH_PATH
STEP4_REFERENCE_PROCESSED_PATH = DATA_PROCESSED_DIR / 'kanagawa_drive_step3' / 'kanagawa_drive_processed.graphml'

STEP4_ROUTE_POINTS = [
    {
        'route_name': 'Kawasaki Station to Great Buddha of Kamakura',
        'start_name': 'Kawasaki Station',
        'start_lat': 35.5314,
        'start_lon': 139.6969,
        'end_name': 'Great Buddha of Kamakura',
        'end_lat': 35.3167,
        'end_lon': 139.5354,
    },
    {
        'route_name': 'Yokohama Station to Odawara Station',
        'start_name': 'Yokohama Station',
        'start_lat': 35.4662,
        'start_lon': 139.6227,
        'end_name': 'Odawara Station',
        'end_lat': 35.2556,
        'end_lon': 139.1597,
    },
    {
        'route_name': 'Fujisawa Station to Hakone-Yumoto Station',
        'start_name': 'Fujisawa Station',
        'start_lat': 35.3388,
        'start_lon': 139.4876,
        'end_name': 'Hakone-Yumoto Station',
        'end_lat': 35.2323,
        'end_lon': 139.1069,
    },
]


def append_step4_log(message):
    with STEP4_LOG_PATH.open('a', encoding='utf-8') as log_file:
        log_file.write(message + '\n')


def graph_stats(graph):
    largest_component_nodes = max(nx.weakly_connected_components(graph), key=len)
    return {
        'node_count': graph.number_of_nodes(),
        'edge_count': graph.number_of_edges(),
        'weak_component_count': nx.number_weakly_connected_components(graph),
        'largest_component_node_count': len(largest_component_nodes),
    }


def safe_stat_mb(path):
    return round(path.stat().st_size / (1024 * 1024), 2) if path.exists() else pd.NA


def nearest_node_without_optional_deps(graph, x_coord, y_coord):
    nearest_node = None
    nearest_distance_sq = None
    for node_id, data in graph.nodes(data=True):
        dx = float(data['x']) - x_coord
        dy = float(data['y']) - y_coord
        distance_sq = (dx * dx) + (dy * dy)
        if nearest_distance_sq is None or distance_sq < nearest_distance_sq:
            nearest_distance_sq = distance_sq
            nearest_node = node_id
    return nearest_node


append_step4_log('Step 4 merge started')
append_step4_log(f"Created at: {datetime.now().isoformat()}")
append_step4_log(f"Input directory: {STEP4_INPUT_DIR}")
append_step4_log(f"Output graph path: {STEP4_OUTPUT_GRAPH_PATH}")
append_step4_log(f"Reference raw path: {STEP4_REFERENCE_RAW_PATH}")
append_step4_log(f"Reference processed path: {STEP4_REFERENCE_PROCESSED_PATH}")

municipality_graph_paths = sorted(STEP4_INPUT_DIR.glob('*.graphml'))
if not municipality_graph_paths:
    raise FileNotFoundError(f'結合対象 GraphML が見つかりません: {STEP4_INPUT_DIR}')

merge_start = time.perf_counter()
merged_graph = nx.MultiDiGraph()
loaded_count = 0

for graph_path in municipality_graph_paths:
    municipality_graph = ox.load_graphml(graph_path)
    merged_graph = nx.compose(merged_graph, municipality_graph)
    loaded_count += 1
    append_step4_log(f"Loaded and merged: {graph_path.name}")

merged_graph = ox.add_edge_speeds(merged_graph)
merged_graph = ox.add_edge_travel_times(merged_graph)
ox.save_graphml(merged_graph, STEP4_OUTPUT_GRAPH_PATH)
reloaded_merged_graph = ox.load_graphml(STEP4_OUTPUT_GRAPH_PATH)
projected_merged_graph = ox.project_graph(merged_graph)
projected_crs = projected_merged_graph.graph['crs']
projected_crs_value = projected_crs.to_string() if hasattr(projected_crs, 'to_string') else str(projected_crs)
to_projected = Transformer.from_crs('EPSG:4326', projected_crs_value, always_xy=True)
merge_elapsed_seconds = round(time.perf_counter() - merge_start, 2)

raw_reference_graph = ox.load_graphml(STEP4_REFERENCE_RAW_PATH)
processed_reference_graph = ox.load_graphml(STEP4_REFERENCE_PROCESSED_PATH) if STEP4_REFERENCE_PROCESSED_PATH.exists() else None

merged_stats = graph_stats(merged_graph)
reloaded_stats = graph_stats(reloaded_merged_graph)
raw_reference_stats = graph_stats(raw_reference_graph)
processed_reference_stats = graph_stats(processed_reference_graph) if processed_reference_graph is not None else {}

route_records = []
for route in STEP4_ROUTE_POINTS:
    try:
        start_x, start_y = to_projected.transform(route['start_lon'], route['start_lat'])
        end_x, end_y = to_projected.transform(route['end_lon'], route['end_lat'])
        start_node = nearest_node_without_optional_deps(projected_merged_graph, start_x, start_y)
        end_node = nearest_node_without_optional_deps(projected_merged_graph, end_x, end_y)
        route_nodes = nx.shortest_path(merged_graph, start_node, end_node, weight='length')
        route_length_m = nx.path_weight(merged_graph, route_nodes, weight='length')
        route_records.append(
            {
                'route_name': route['route_name'],
                'start_name': route['start_name'],
                'end_name': route['end_name'],
                'start_node': start_node,
                'end_node': end_node,
                'path_node_count': len(route_nodes),
                'route_length_m': round(route_length_m, 2),
                'route_length_km': round(route_length_m / 1000, 2),
                'status': 'success',
                'error_message': '',
            }
        )
        append_step4_log(
            f"ROUTE SUCCESS {route['route_name']} length_km={round(route_length_m / 1000, 2)} nodes={len(route_nodes)}"
        )
    except Exception as exc:
        error_message = str(exc).replace('\n', ' ')[:1000]
        route_records.append(
            {
                'route_name': route['route_name'],
                'start_name': route['start_name'],
                'end_name': route['end_name'],
                'start_node': pd.NA,
                'end_node': pd.NA,
                'path_node_count': pd.NA,
                'route_length_m': pd.NA,
                'route_length_km': pd.NA,
                'status': 'error',
                'error_message': error_message,
            }
        )
        append_step4_log(f"ROUTE ERROR {route['route_name']}: {error_message}")

route_checks_df = pd.DataFrame(route_records)
route_checks_df.to_csv(STEP4_ROUTE_PATH, index=False, encoding='utf-8-sig')

merged_node_count = merged_stats['node_count']
merged_edge_count = merged_stats['edge_count']
merged_component_count = merged_stats['weak_component_count']
raw_reference_node_count = raw_reference_stats['node_count']
raw_reference_edge_count = raw_reference_stats['edge_count']
processed_reference_node_count = processed_reference_stats.get('node_count', pd.NA)
processed_reference_edge_count = processed_reference_stats.get('edge_count', pd.NA)
successful_route_checks = int((route_checks_df['status'] == 'success').sum())
failed_route_checks = int((route_checks_df['status'] == 'error').sum())

metrics_step4_df = pd.DataFrame(
    [
        {
            'loaded_graph_count': loaded_count,
            'merged_graph_path': str(STEP4_OUTPUT_GRAPH_PATH),
            'merged_file_size_mb': safe_stat_mb(STEP4_OUTPUT_GRAPH_PATH),
            'elapsed_seconds': merge_elapsed_seconds,
            'merged_node_count': merged_node_count,
            'merged_edge_count': merged_edge_count,
            'merged_weak_component_count': merged_component_count,
            'merged_largest_component_node_count': merged_stats['largest_component_node_count'],
            'reloaded_node_count': reloaded_stats['node_count'],
            'reloaded_edge_count': reloaded_stats['edge_count'],
            'raw_reference_node_count': raw_reference_node_count,
            'raw_reference_edge_count': raw_reference_edge_count,
            'raw_reference_file_size_mb': safe_stat_mb(STEP4_REFERENCE_RAW_PATH),
            'processed_reference_node_count': processed_reference_node_count,
            'processed_reference_edge_count': processed_reference_edge_count,
            'processed_reference_file_size_mb': safe_stat_mb(STEP4_REFERENCE_PROCESSED_PATH),
            'delta_nodes_vs_raw': merged_node_count - raw_reference_node_count,
            'delta_edges_vs_raw': merged_edge_count - raw_reference_edge_count,
            'delta_nodes_vs_processed': merged_node_count - processed_reference_node_count if processed_reference_graph is not None else pd.NA,
            'delta_edges_vs_processed': merged_edge_count - processed_reference_edge_count if processed_reference_graph is not None else pd.NA,
            'successful_route_checks': successful_route_checks,
            'failed_route_checks': failed_route_checks,
        }
    ]
)
metrics_step4_df.to_csv(STEP4_METRICS_PATH, index=False, encoding='utf-8-sig')

append_step4_log(f"Loaded graph count: {loaded_count}")
append_step4_log(f"Merged node count: {merged_node_count}")
append_step4_log(f"Merged edge count: {merged_edge_count}")
append_step4_log(f"Merged weak component count: {merged_component_count}")
append_step4_log(f"Elapsed seconds: {merge_elapsed_seconds}")
append_step4_log(f"Merged file size (MB): {safe_stat_mb(STEP4_OUTPUT_GRAPH_PATH)}")
append_step4_log(f"Route check successes: {successful_route_checks}")
append_step4_log(f"Route check errors: {failed_route_checks}")
append_step4_log('Step 4 merge completed')

print(f"Loaded graph count: {loaded_count}")
print(f"Merged graph path: {STEP4_OUTPUT_GRAPH_PATH}")
print(f"Metrics path: {STEP4_METRICS_PATH}")
print(f"Route checks path: {STEP4_ROUTE_PATH}")
print(f"Log path: {STEP4_LOG_PATH}")
print(f"Merged nodes: {merged_node_count}")
print(f"Merged edges: {merged_edge_count}")
print(f"Merged weak components: {merged_component_count}")
print(f"Reloaded nodes: {reloaded_stats['node_count']}")
print(f"Reloaded edges: {reloaded_stats['edge_count']}")
print(f"Raw reference nodes: {raw_reference_node_count}")
print(f"Raw reference edges: {raw_reference_edge_count}")
if processed_reference_graph is not None:
    print(f"Processed reference nodes: {processed_reference_node_count}")
    print(f"Processed reference edges: {processed_reference_edge_count}")
print(f"Delta nodes vs raw: {merged_node_count - raw_reference_node_count}")
print(f"Delta edges vs raw: {merged_edge_count - raw_reference_edge_count}")
print(f"Merged file size (MB): {safe_stat_mb(STEP4_OUTPUT_GRAPH_PATH)}")
print(f"Elapsed seconds: {merge_elapsed_seconds}")
print(route_checks_df[['route_name', 'status', 'route_length_km', 'path_node_count', 'error_message']].to_string(index=False))

metrics_step4_df


Loaded graph count: 33
Merged graph path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\processed\kanagawa_municipality_merged.graphml
Metrics path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_municipality_merged_metrics.csv
Route checks path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_municipality_merged_route_checks.csv
Log path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\logs\001_osm_network_feasibility_validation_plan_step4_20260319_104935.log
Merged nodes: 223152
Merged edges: 604442
Merged weak components: 21
Reloaded nodes: 223152
Reloaded edges: 604442
Raw reference nodes: 223850
Raw reference edges: 608205
Processed reference nodes: 223850
Processed reference edges: 608205
Delta nodes vs raw: -698
Delta edges vs raw: -3763
Mer

,loaded_graph_count,merged_graph_path,merged_file_size_mb,elapsed_seconds,merged_node_count,merged_edge_count,merged_weak_component_count,merged_largest_component_node_count,reloaded_node_count,reloaded_edge_count,...,raw_reference_file_size_mb,processed_reference_node_count,processed_reference_edge_count,processed_reference_file_size_mb,delta_nodes_vs_raw,delta_edges_vs_raw,delta_nodes_vs_processed,delta_edges_vs_processed,successful_route_checks,failed_route_checks
0,33,C:\Users\LCCadmin\OneDrive - 株式会社 ロジクロス・コミュニケー...,310.64,332.17,223152,604442,21,153806,223152,604442,...,259.52,223850,608205,285.75,-698,-3763,-698,-3763,1,2


# 結合後ネットワークの原因究明

Step 4 の結合結果で発生した差分と経路未疎通の原因を調べる。

## 実行内容

1. 結合後グラフの弱連結成分数とサイズ分布を確認する
2. 有名地点がどの連結成分に属しているかを確認する
3. 一括取得版と結合版のノード・エッジ差分件数を確認する
4. 最大連結成分に含まれない代表ノードの座標を確認する
5. 後続の結合改善に向けた調査材料を残す

## 出力想定内容

- 連結成分のサイズ一覧
- 主要地点ごとの所属成分
- ノード差分数、エッジ差分数
- 分断された成分の代表位置


In [15]:
STEP4_ANALYSIS_PATH = OUTPUT_METRICS_DIR / 'kanagawa_municipality_merged_component_analysis.csv'
STEP4_ANALYSIS_ROUTE_PATH = OUTPUT_METRICS_DIR / 'kanagawa_municipality_merged_component_route_points.csv'

analysis_graph = ox.load_graphml(STEP4_OUTPUT_GRAPH_PATH)
analysis_raw_graph = ox.load_graphml(STEP4_REFERENCE_RAW_PATH)

components = list(nx.weakly_connected_components(analysis_graph))
component_rows = []
node_to_component = {}

for component_index, component_nodes in enumerate(sorted(components, key=len, reverse=True), start=1):
    subgraph = analysis_graph.subgraph(component_nodes)
    sample_node = next(iter(component_nodes))
    sample_data = analysis_graph.nodes[sample_node]
    component_rows.append(
        {
            'component_id': component_index,
            'node_count': subgraph.number_of_nodes(),
            'edge_count': subgraph.number_of_edges(),
            'sample_node': sample_node,
            'sample_lon': float(sample_data['x']),
            'sample_lat': float(sample_data['y']),
        }
    )
    for node_id in component_nodes:
        node_to_component[node_id] = component_index

component_df = pd.DataFrame(component_rows)
component_df.to_csv(STEP4_ANALYSIS_PATH, index=False, encoding='utf-8-sig')

projected_analysis_graph = ox.project_graph(analysis_graph)
analysis_projected_crs = projected_analysis_graph.graph['crs']
analysis_projected_crs_value = analysis_projected_crs.to_string() if hasattr(analysis_projected_crs, 'to_string') else str(analysis_projected_crs)
analysis_transformer = Transformer.from_crs('EPSG:4326', analysis_projected_crs_value, always_xy=True)

route_point_rows = []
for route in STEP4_ROUTE_POINTS:
    for point_type in ['start', 'end']:
        point_name = route[f'{point_type}_name']
        point_lat = route[f'{point_type}_lat']
        point_lon = route[f'{point_type}_lon']
        point_x, point_y = analysis_transformer.transform(point_lon, point_lat)
        nearest_node = nearest_node_without_optional_deps(projected_analysis_graph, point_x, point_y)
        route_point_rows.append(
            {
                'route_name': route['route_name'],
                'point_type': point_type,
                'point_name': point_name,
                'nearest_node': nearest_node,
                'component_id': node_to_component.get(nearest_node, pd.NA),
                'lon': point_lon,
                'lat': point_lat,
            }
        )

route_point_df = pd.DataFrame(route_point_rows)
route_point_df.to_csv(STEP4_ANALYSIS_ROUTE_PATH, index=False, encoding='utf-8-sig')

merged_nodes = set(analysis_graph.nodes)
raw_nodes = set(analysis_raw_graph.nodes)
missing_node_count_vs_raw = len(raw_nodes - merged_nodes)
extra_node_count_vs_raw = len(merged_nodes - raw_nodes)

merged_edges = set(analysis_graph.edges(keys=True))
raw_edges = set(analysis_raw_graph.edges(keys=True))
missing_edge_count_vs_raw = len(raw_edges - merged_edges)
extra_edge_count_vs_raw = len(merged_edges - raw_edges)

print(f'Weak component count: {len(component_df)}')
print(component_df[['component_id', 'node_count', 'edge_count', 'sample_lon', 'sample_lat']].to_string(index=False))
print('')
print('Route point component mapping:')
print(route_point_df.to_string(index=False))
print('')
print(f'Missing nodes vs raw: {missing_node_count_vs_raw}')
print(f'Extra nodes vs raw: {extra_node_count_vs_raw}')
print(f'Missing edges vs raw: {missing_edge_count_vs_raw}')
print(f'Extra edges vs raw: {extra_edge_count_vs_raw}')

component_df


Weak component count: 21
 component_id  node_count  edge_count  sample_lon  sample_lat
            1      153806      423403  139.507311   35.451334
            2       16742       43313  139.317160   35.497278
            3       11558       30361  139.671966   35.271486
            4        9298       22676  139.185298   35.367255
            5        9093       25336  139.308408   35.367958
            6        6301       16772  139.218918   35.310350
            7        2588        6522  139.325283   35.525081
            8        1983        5019  139.600515   35.301955
            9        1831        4838  139.641214   35.161674
           10        1474        3937  139.124332   35.163622
           11        1344        3314  139.587585   35.270943
           12        1244        3468  139.091584   35.348870
           13        1100        2949  139.267543   35.301508
           14        1039        2687  139.057863   35.244514
           15         806        2163  139.23

,component_id,node_count,edge_count,sample_node,sample_lon,sample_lat
0,1,153806,423403,1559186922,139.507311,35.451334
1,2,16742,43313,6904260748,139.317160,35.497278
2,3,11558,30361,392658952,139.671966,35.271486
3,4,9298,22676,3226763264,139.185298,35.367255
4,5,9093,25336,6715670534,139.308408,35.367958
5,6,6301,16772,454164532,139.218918,35.310350
6,7,2588,6522,277225472,139.325283,35.525081
7,8,1983,5019,1771937798,139.600515,35.301955
8,9,1831,4838,354828290,139.641214,35.161674
9,10,1474,3937,9310584832,139.124332,35.163622


# 境界補完接続の試行

結合後グラフで分断されている `component 1`, `component 6`, `component 14` を対象に、境界付近のノード同士を仮想エッジで補完接続した場合に、経路未疎通が改善するかを検証する。

## 実行内容

1. `component 1`, `component 6`, `component 14` のノードを抽出する
2. `10m`, `20m`, `30m`, `50m` のしきい値ごとに最近傍候補を探索する
3. 距離だけでなく、`highway`, `oneway`, `maxspeed` の整合を見て接続候補を絞る
4. 候補ノード間に仮想エッジを追加し、元データと区別できる属性を付与する
5. 補完後グラフで有名地点間の経路が `NA` にならないかを確認する
6. しきい値ごとの接続本数、経路成否、距離を記録する

## 出力想定内容

- しきい値ごとの補完接続候補一覧
- 補完後の経路疎通結果
- どのしきい値で `NA` が解消するか
- 一括取得との差分確認に進める候補しきい値

## 補足

- 補完エッジには `is_boundary_bridge=True` を付けて元データと区別する
- 今回は PoC 検証として、距離と道路属性の中間条件で接続候補を選別する
- 許容差分の定義は次段で仕分けるが、まずは `NA` が出ないことを優先する


In [17]:
from collections import Counter
from sklearn.neighbors import BallTree

STEP4_BRIDGE_THRESHOLDS_M = [10, 20, 30, 50]
STEP4_TARGET_COMPONENTS = [1, 6, 14]
STEP4_BRIDGE_CANDIDATES_PATH = OUTPUT_METRICS_DIR / 'kanagawa_municipality_boundary_bridge_candidates.csv'
STEP4_BRIDGE_RESULTS_PATH = OUTPUT_METRICS_DIR / 'kanagawa_municipality_boundary_bridge_results.csv'


def first_scalar(value):
    if isinstance(value, (list, tuple, set)):
        for item in value:
            if item is None:
                continue
            if pd.isna(item):
                continue
            if str(item).strip() == '':
                continue
            return str(item)
        return ''
    if value is None:
        return ''
    if pd.isna(value):
        return ''
    return str(value)


def normalize_highway(value):
    text = first_scalar(value)
    return text.split(',')[0].strip().lower() if text else ''


def normalize_oneway(value):
    text = first_scalar(value).strip().lower()
    if text in {'true', 'yes', '1'}:
        return 'yes'
    if text in {'false', 'no', '0'}:
        return 'no'
    return ''


def parse_maxspeed(value):
    text = first_scalar(value)
    digits = ''.join(ch for ch in text if ch.isdigit())
    return float(digits) if digits else None


def node_context(graph, node_id):
    highways = []
    oneways = []
    maxspeeds = []
    for _, _, _, edge_data in graph.edges(node_id, keys=True, data=True):
        highway = normalize_highway(edge_data.get('highway', ''))
        oneway = normalize_oneway(edge_data.get('oneway', ''))
        maxspeed = parse_maxspeed(edge_data.get('maxspeed', ''))
        if highway:
            highways.append(highway)
        if oneway:
            oneways.append(oneway)
        if maxspeed is not None:
            maxspeeds.append(maxspeed)
    highway_mode = Counter(highways).most_common(1)[0][0] if highways else ''
    oneway_mode = Counter(oneways).most_common(1)[0][0] if oneways else ''
    maxspeed_median = float(pd.Series(maxspeeds).median()) if maxspeeds else None
    return {
        'highway': highway_mode,
        'oneway': oneway_mode,
        'maxspeed': maxspeed_median,
    }


def compatible_context(left_context, right_context):
    left_highway = left_context['highway']
    right_highway = right_context['highway']
    if left_highway and right_highway and left_highway != right_highway:
        return False
    left_oneway = left_context['oneway']
    right_oneway = right_context['oneway']
    if left_oneway and right_oneway and left_oneway != right_oneway:
        return False
    left_speed = left_context['maxspeed']
    right_speed = right_context['maxspeed']
    if left_speed is not None and right_speed is not None and abs(left_speed - right_speed) > 30:
        return False
    return True


def average_speed_kph(left_context, right_context):
    speeds = [speed for speed in [left_context['maxspeed'], right_context['maxspeed']] if speed is not None]
    return round(sum(speeds) / len(speeds), 2) if speeds else 30.0


bridge_graph = ox.load_graphml(STEP4_OUTPUT_GRAPH_PATH)
bridge_projected_graph = ox.project_graph(bridge_graph)
bridge_projected_crs = bridge_projected_graph.graph['crs']
bridge_projected_crs_value = bridge_projected_crs.to_string() if hasattr(bridge_projected_crs, 'to_string') else str(bridge_projected_crs)
bridge_transformer = Transformer.from_crs('EPSG:4326', bridge_projected_crs_value, always_xy=True)

bridge_components = list(nx.weakly_connected_components(bridge_graph))
sorted_bridge_components = sorted(bridge_components, key=len, reverse=True)
component_id_to_nodes = {index: nodes for index, nodes in enumerate(sorted_bridge_components, start=1)}

component_xy = {}
component_context = {}
for component_id in STEP4_TARGET_COMPONENTS:
    node_ids = list(component_id_to_nodes[component_id])
    xy_rows = []
    context_map = {}
    for node_id in node_ids:
        data = bridge_projected_graph.nodes[node_id]
        xy_rows.append((node_id, float(data['x']), float(data['y'])))
        context_map[node_id] = node_context(bridge_graph, node_id)
    component_xy[component_id] = pd.DataFrame(xy_rows, columns=['node_id', 'x', 'y'])
    component_context[component_id] = context_map

base_component_df = component_xy[1]
base_tree = BallTree(base_component_df[['y', 'x']].to_numpy(), metric='euclidean')

candidate_rows = []
result_rows = []

for threshold_m in STEP4_BRIDGE_THRESHOLDS_M:
    threshold_graph = bridge_graph.copy()
    threshold_candidate_pairs = []
    for component_id in [6, 14]:
        target_df = component_xy[component_id]
        distances, indices = base_tree.query(target_df[['y', 'x']].to_numpy(), k=3)
        chosen_pairs = []
        used_base_nodes = set()
        for row_index, target_row in target_df.iterrows():
            target_node = target_row['node_id']
            target_context = component_context[component_id][target_node]
            for distance_value, nearest_index in zip(distances[row_index], indices[row_index]):
                euclidean_distance_m = round(float(distance_value), 2)
                base_row = base_component_df.iloc[int(nearest_index)]
                base_node = base_row['node_id']
                if euclidean_distance_m > threshold_m:
                    continue
                if base_node in used_base_nodes:
                    continue
                base_context = component_context[1][base_node]
                if not compatible_context(base_context, target_context):
                    continue
                candidate_rows.append(
                    {
                        'threshold_m': threshold_m,
                        'from_component_id': 1,
                        'to_component_id': component_id,
                        'base_node': base_node,
                        'target_node': target_node,
                        'distance_m': euclidean_distance_m,
                        'base_highway': base_context['highway'],
                        'target_highway': target_context['highway'],
                        'base_oneway': base_context['oneway'],
                        'target_oneway': target_context['oneway'],
                        'base_maxspeed': base_context['maxspeed'],
                        'target_maxspeed': target_context['maxspeed'],
                    }
                )
                chosen_pairs.append((base_node, target_node, euclidean_distance_m, base_context, target_context, component_id))
                used_base_nodes.add(base_node)
                break
            if len(chosen_pairs) >= 5:
                break
        threshold_candidate_pairs.extend(chosen_pairs)

    for base_node, target_node, distance_m, base_context, target_context, component_id in threshold_candidate_pairs:
        bridge_speed_kph = average_speed_kph(base_context, target_context)
        bridge_travel_time_sec = round(distance_m / (bridge_speed_kph * 1000 / 3600), 2)
        bridge_attrs = {
            'length': distance_m,
            'travel_time': bridge_travel_time_sec,
            'maxspeed': bridge_speed_kph,
            'highway': base_context['highway'] or target_context['highway'] or 'boundary_bridge',
            'oneway': False,
            'name': f'boundary_bridge_1_{component_id}_{threshold_m}m',
            'is_boundary_bridge': True,
            'bridge_threshold_m': threshold_m,
            'bridge_distance_m': distance_m,
        }
        threshold_graph.add_edge(base_node, target_node, **bridge_attrs)
        threshold_graph.add_edge(target_node, base_node, **bridge_attrs)

    threshold_projected_graph = ox.project_graph(threshold_graph)
    threshold_projected_crs = threshold_projected_graph.graph['crs']
    threshold_projected_crs_value = threshold_projected_crs.to_string() if hasattr(threshold_projected_crs, 'to_string') else str(threshold_projected_crs)
    threshold_transformer = Transformer.from_crs('EPSG:4326', threshold_projected_crs_value, always_xy=True)

    for route in STEP4_ROUTE_POINTS:
        try:
            start_x, start_y = threshold_transformer.transform(route['start_lon'], route['start_lat'])
            end_x, end_y = threshold_transformer.transform(route['end_lon'], route['end_lat'])
            start_node = nearest_node_without_optional_deps(threshold_projected_graph, start_x, start_y)
            end_node = nearest_node_without_optional_deps(threshold_projected_graph, end_x, end_y)
            route_nodes = nx.shortest_path(threshold_graph, start_node, end_node, weight='length')
            route_length_m = nx.path_weight(threshold_graph, route_nodes, weight='length')
            result_rows.append(
                {
                    'threshold_m': threshold_m,
                    'route_name': route['route_name'],
                    'status': 'success',
                    'route_length_km': round(route_length_m / 1000, 2),
                    'path_node_count': len(route_nodes),
                    'bridge_edge_pairs_added': len(threshold_candidate_pairs),
                    'error_message': '',
                }
            )
        except Exception as exc:
            result_rows.append(
                {
                    'threshold_m': threshold_m,
                    'route_name': route['route_name'],
                    'status': 'error',
                    'route_length_km': pd.NA,
                    'path_node_count': pd.NA,
                    'bridge_edge_pairs_added': len(threshold_candidate_pairs),
                    'error_message': str(exc).replace('\n', ' ')[:1000],
                }
            )

bridge_candidates_df = pd.DataFrame(candidate_rows)
bridge_results_df = pd.DataFrame(result_rows)
bridge_candidates_df.to_csv(STEP4_BRIDGE_CANDIDATES_PATH, index=False, encoding='utf-8-sig')
bridge_results_df.to_csv(STEP4_BRIDGE_RESULTS_PATH, index=False, encoding='utf-8-sig')

print('Bridge candidate summary:')
if bridge_candidates_df.empty:
    print('No compatible bridge candidates were found.')
else:
    print(bridge_candidates_df[['threshold_m', 'to_component_id', 'distance_m', 'base_highway', 'target_highway', 'base_oneway', 'target_oneway', 'base_maxspeed', 'target_maxspeed']].to_string(index=False))
print('')
print('Bridge route results:')
print(bridge_results_df.to_string(index=False))

bridge_results_df


Bridge candidate summary:
No compatible bridge candidates were found.

Bridge route results:
 threshold_m                                   route_name  status route_length_km path_node_count  bridge_edge_pairs_added                              error_message
          10 Kawasaki Station to Great Buddha of Kamakura success           33.87             373                        0                                           
          10          Yokohama Station to Odawara Station   error            <NA>            <NA>                        0 No path between 8120975721 and 2451183506.
          10    Fujisawa Station to Hakone-Yumoto Station   error            <NA>            <NA>                        0 No path between 2398078142 and 6138273291.
          20 Kawasaki Station to Great Buddha of Kamakura success           33.87             373                        0                                           
          20          Yokohama Station to Odawara Station   error            

,threshold_m,route_name,status,route_length_km,path_node_count,bridge_edge_pairs_added,error_message
0,10,Kawasaki Station to Great Buddha of Kamakura,success,33.87,373,0,
1,10,Yokohama Station to Odawara Station,error,<NA>,<NA>,0,No path between 8120975721 and 2451183506.
2,10,Fujisawa Station to Hakone-Yumoto Station,error,<NA>,<NA>,0,No path between 2398078142 and 6138273291.
3,20,Kawasaki Station to Great Buddha of Kamakura,success,33.87,373,0,
4,20,Yokohama Station to Odawara Station,error,<NA>,<NA>,0,No path between 8120975721 and 2451183506.
5,20,Fujisawa Station to Hakone-Yumoto Station,error,<NA>,<NA>,0,No path between 2398078142 and 6138273291.
6,30,Kawasaki Station to Great Buddha of Kamakura,success,33.87,373,0,
7,30,Yokohama Station to Odawara Station,error,<NA>,<NA>,0,No path between 8120975721 and 2451183506.
8,30,Fujisawa Station to Hakone-Yumoto Station,error,<NA>,<NA>,0,No path between 2398078142 and 6138273291.
9,50,Kawasaki Station to Great Buddha of Kamakura,success,33.87,373,0,


# 補完候補の切り分け分析

補完接続が成立しない理由が、候補ノード自体が存在しないためか、道路属性条件で落ちているためかを切り分ける。

## 実行内容

1. `component 1-6` と `component 1-14` を対象に近傍候補を探索する
2. 距離条件だけを満たす候補件数を数える
3. `highway`, `oneway`, `maxspeed` 条件適用後の候補件数を数える
4. 条件で落ちた理由を集計する
5. どこを緩めるべきか、そもそも候補がないのかを判断する材料を出す

## 出力想定内容

- 距離のみ候補件数
- 条件適用後候補件数
- 落選理由ごとの件数
- 代表的な候補ノード対


In [18]:
STEP4_BRIDGE_DIAGNOSIS_PATH = OUTPUT_METRICS_DIR / 'kanagawa_municipality_boundary_bridge_diagnosis.csv'
STEP4_BRIDGE_DIAGNOSIS_SAMPLE_PATH = OUTPUT_METRICS_DIR / 'kanagawa_municipality_boundary_bridge_diagnosis_samples.csv'


def incompatibility_reason(left_context, right_context, distance_m, threshold_m):
    if distance_m > threshold_m:
        return 'distance_over_threshold'
    left_highway = left_context['highway']
    right_highway = right_context['highway']
    if left_highway and right_highway and left_highway != right_highway:
        return 'highway_mismatch'
    left_oneway = left_context['oneway']
    right_oneway = right_context['oneway']
    if left_oneway and right_oneway and left_oneway != right_oneway:
        return 'oneway_mismatch'
    left_speed = left_context['maxspeed']
    right_speed = right_context['maxspeed']
    if left_speed is not None and right_speed is not None and abs(left_speed - right_speed) > 30:
        return 'maxspeed_mismatch'
    return 'compatible'


diagnosis_rows = []
diagnosis_sample_rows = []

diagnosis_component_pairs = [(1, 6), (1, 14)]
for threshold_m in STEP4_BRIDGE_THRESHOLDS_M:
    for base_component_id, target_component_id in diagnosis_component_pairs:
        base_df = component_xy[base_component_id]
        target_df = component_xy[target_component_id]
        base_tree = BallTree(base_df[['y', 'x']].to_numpy(), metric='euclidean')
        distances, indices = base_tree.query(target_df[['y', 'x']].to_numpy(), k=5)

        counters = Counter()
        compatible_examples = 0
        for row_index, target_row in target_df.iterrows():
            target_node = target_row['node_id']
            target_context = component_context[target_component_id][target_node]
            for distance_value, nearest_index in zip(distances[row_index], indices[row_index]):
                base_row = base_df.iloc[int(nearest_index)]
                base_node = base_row['node_id']
                base_context = component_context[base_component_id][base_node]
                distance_m = round(float(distance_value), 2)
                reason = incompatibility_reason(base_context, target_context, distance_m, threshold_m)
                counters[reason] += 1
                if len(diagnosis_sample_rows) < 300:
                    diagnosis_sample_rows.append(
                        {
                            'threshold_m': threshold_m,
                            'base_component_id': base_component_id,
                            'target_component_id': target_component_id,
                            'base_node': base_node,
                            'target_node': target_node,
                            'distance_m': distance_m,
                            'reason': reason,
                            'base_highway': base_context['highway'],
                            'target_highway': target_context['highway'],
                            'base_oneway': base_context['oneway'],
                            'target_oneway': target_context['oneway'],
                            'base_maxspeed': base_context['maxspeed'],
                            'target_maxspeed': target_context['maxspeed'],
                        }
                    )
                if reason == 'compatible':
                    compatible_examples += 1

        diagnosis_rows.append(
            {
                'threshold_m': threshold_m,
                'base_component_id': base_component_id,
                'target_component_id': target_component_id,
                'distance_only_candidates': counters['compatible'] + counters['highway_mismatch'] + counters['oneway_mismatch'] + counters['maxspeed_mismatch'],
                'compatible_candidates': counters['compatible'],
                'highway_mismatch': counters['highway_mismatch'],
                'oneway_mismatch': counters['oneway_mismatch'],
                'maxspeed_mismatch': counters['maxspeed_mismatch'],
                'distance_over_threshold': counters['distance_over_threshold'],
                'evaluated_pairs': sum(counters.values()),
            }
        )

bridge_diagnosis_df = pd.DataFrame(diagnosis_rows)
bridge_diagnosis_sample_df = pd.DataFrame(diagnosis_sample_rows)
bridge_diagnosis_df.to_csv(STEP4_BRIDGE_DIAGNOSIS_PATH, index=False, encoding='utf-8-sig')
bridge_diagnosis_sample_df.to_csv(STEP4_BRIDGE_DIAGNOSIS_SAMPLE_PATH, index=False, encoding='utf-8-sig')

print('Bridge candidate diagnosis summary:')
print(bridge_diagnosis_df.to_string(index=False))
print('')
print('Bridge candidate diagnosis samples:')
print(bridge_diagnosis_sample_df.head(40).to_string(index=False))

bridge_diagnosis_df


Bridge candidate diagnosis summary:
 threshold_m  base_component_id  target_component_id  distance_only_candidates  compatible_candidates  highway_mismatch  oneway_mismatch  maxspeed_mismatch  distance_over_threshold  evaluated_pairs
          10                  1                    6                         0                      0                 0                0                  0                    31505            31505
          10                  1                   14                         0                      0                 0                0                  0                     5195             5195
          20                  1                    6                         0                      0                 0                0                  0                    31505            31505
          20                  1                   14                         0                      0                 0                0                  0                 

,threshold_m,base_component_id,target_component_id,distance_only_candidates,compatible_candidates,highway_mismatch,oneway_mismatch,maxspeed_mismatch,distance_over_threshold,evaluated_pairs
0,10,1,6,0,0,0,0,0,31505,31505
1,10,1,14,0,0,0,0,0,5195,5195
2,20,1,6,0,0,0,0,0,31505,31505
3,20,1,14,0,0,0,0,0,5195,5195
4,30,1,6,0,0,0,0,0,31505,31505
5,30,1,14,0,0,0,0,0,5195,5195
6,50,1,6,0,0,0,0,0,31505,31505
7,50,1,14,0,0,0,0,0,5195,5195


# 一括取得版と分割結合版の経路比較

同じ有名地点の組み合わせについて、一括取得版と分割結合版の両方で経路が取得できるかを比較する。

## 実行内容

1. 一括取得版グラフを読み込む
2. 分割結合版グラフを読み込む
3. 同じ地点ペアで最寄りノードと最短経路を確認する
4. 取得可否と距離を `print()` する
5. 比較結果を CSV に保存する

## 出力想定内容

- 一括取得版では通るか
- 分割結合版では通るか
- どのルートで差が出るか
- 分割結合方式を継続するか見直すかの判断材料


In [19]:
STEP4_BATCH_VS_SPLIT_PATH = OUTPUT_METRICS_DIR / 'kanagawa_batch_vs_split_route_checks.csv'


def projected_route_check(graph, route_points):
    projected_graph = ox.project_graph(graph)
    projected_crs = projected_graph.graph['crs']
    projected_crs_value = projected_crs.to_string() if hasattr(projected_crs, 'to_string') else str(projected_crs)
    transformer = Transformer.from_crs('EPSG:4326', projected_crs_value, always_xy=True)
    records = []
    for route in route_points:
        try:
            start_x, start_y = transformer.transform(route['start_lon'], route['start_lat'])
            end_x, end_y = transformer.transform(route['end_lon'], route['end_lat'])
            start_node = nearest_node_without_optional_deps(projected_graph, start_x, start_y)
            end_node = nearest_node_without_optional_deps(projected_graph, end_x, end_y)
            route_nodes = nx.shortest_path(graph, start_node, end_node, weight='length')
            route_length_m = nx.path_weight(graph, route_nodes, weight='length')
            records.append(
                {
                    'route_name': route['route_name'],
                    'status': 'success',
                    'route_length_km': round(route_length_m / 1000, 2),
                    'path_node_count': len(route_nodes),
                    'error_message': '',
                }
            )
        except Exception as exc:
            records.append(
                {
                    'route_name': route['route_name'],
                    'status': 'error',
                    'route_length_km': pd.NA,
                    'path_node_count': pd.NA,
                    'error_message': str(exc).replace('\n', ' ')[:1000],
                }
            )
    return pd.DataFrame(records)


batch_graph = ox.load_graphml(GRAPH_PATH)
split_graph = ox.load_graphml(STEP4_OUTPUT_GRAPH_PATH)

batch_route_df = projected_route_check(batch_graph, STEP4_ROUTE_POINTS)
batch_route_df['graph_type'] = 'batch'
split_route_df = projected_route_check(split_graph, STEP4_ROUTE_POINTS)
split_route_df['graph_type'] = 'split_merged'

batch_vs_split_df = pd.concat([batch_route_df, split_route_df], ignore_index=True)
batch_vs_split_df = batch_vs_split_df[['graph_type', 'route_name', 'status', 'route_length_km', 'path_node_count', 'error_message']]
batch_vs_split_df.to_csv(STEP4_BATCH_VS_SPLIT_PATH, index=False, encoding='utf-8-sig')

print('Batch vs split route comparison:')
print(batch_vs_split_df.to_string(index=False))

batch_vs_split_df


Batch vs split route comparison:
  graph_type                                   route_name  status route_length_km path_node_count                              error_message
       batch Kawasaki Station to Great Buddha of Kamakura success           32.89             359                                           
       batch          Yokohama Station to Odawara Station success           53.54             370                                           
       batch    Fujisawa Station to Hakone-Yumoto Station success           38.88             250                                           
split_merged Kawasaki Station to Great Buddha of Kamakura success           33.87             373                                           
split_merged          Yokohama Station to Odawara Station   error            <NA>            <NA> No path between 8120975721 and 2451183506.
split_merged    Fujisawa Station to Hakone-Yumoto Station   error            <NA>            <NA> No path between 2398078

,graph_type,route_name,status,route_length_km,path_node_count,error_message
0,batch,Kawasaki Station to Great Buddha of Kamakura,success,32.89,359,
1,batch,Yokohama Station to Odawara Station,success,53.54,370,
2,batch,Fujisawa Station to Hakone-Yumoto Station,success,38.88,250,
3,split_merged,Kawasaki Station to Great Buddha of Kamakura,success,33.87,373,
4,split_merged,Yokohama Station to Odawara Station,error,<NA>,<NA>,No path between 8120975721 and 2451183506.
5,split_merged,Fujisawa Station to Hakone-Yumoto Station,error,<NA>,<NA>,No path between 2398078142 and 6138273291.


# 親グラフからの自治体切り出し検証

既存の行政区別個別取得データを後から結合する方式では、県全体 routing に必要な道路接続を保持できなかった。そこで、都道府県一括取得版を親グラフとして保持し、その 1 枚の地図から自治体単位に切り出す方式を追加検証する。

## 実行内容

1. 神奈川県一括取得版 GraphML を親グラフとして読み込む
2. 市区町村一覧をもとに各自治体ポリゴンを取得する
3. 親グラフ上のノードとエッジを、各自治体ポリゴンに交差するものとして切り出す
4. 切り出し結果を自治体別 GraphML として保存する
5. 切り出しグラフを再結合し、一括取得版とのノード数・エッジ数・component 数を比較する
6. 有名地点の経路疎通確認を行い、県全体 routing に使えるかを確認する

## 想定入出力

- 入力 1: `data/raw/road_network/kanagawa_drive.graphml`
- 入力 2: `data/raw/admin_units/kanagawa_municipalities.csv`
- 出力 1: `data/processed/kanagawa_municipalities_from_master/`
- 出力 2: `data/processed/kanagawa_municipality_merged_from_master.graphml`
- 出力 3: 比較メトリクス CSV、経路疎通確認 CSV、実行ログ

## 出力想定内容

- 親グラフ基準で切り出した自治体別 GraphML 群
- 親グラフ再結合版の基本メトリクス
- 一括取得版との差分件数
- 県内有名地点の経路疎通可否

## 補足

- 既存の `graph_from_place` による自治体別個別取得とは別方式である
- この方式では、分割データは一括取得版を正本とした利用用キャッシュとして扱う


In [20]:
STEP45_OUTPUT_DIR = DATA_PROCESSED_DIR / 'kanagawa_municipalities_from_master'
STEP45_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STEP45_MERGED_GRAPH_PATH = DATA_PROCESSED_DIR / 'kanagawa_municipality_merged_from_master.graphml'
STEP45_SLICE_METRICS_PATH = OUTPUT_METRICS_DIR / 'kanagawa_municipalities_from_master_metrics.csv'
STEP45_COMPARE_METRICS_PATH = OUTPUT_METRICS_DIR / 'kanagawa_municipalities_from_master_compare_metrics.csv'
STEP45_ROUTE_CHECKS_PATH = OUTPUT_METRICS_DIR / 'kanagawa_municipalities_from_master_route_checks.csv'
STEP45_LOG_PATH = LOG_DIR / f"001_osm_network_feasibility_validation_plan_step45_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

master_graph = ox.load_graphml(GRAPH_PATH)
admin_units_df = pd.read_csv(ADMIN_UNITS_PATH)

master_nodes_gdf, master_edges_gdf = ox.graph_to_gdfs(master_graph)
master_edges_gdf = master_edges_gdf.copy()
if 'geometry' not in master_edges_gdf.columns:
    master_edges_gdf['geometry'] = master_edges_gdf.apply(lambda row: LineString([(master_graph.nodes[row.name[0]]['x'], master_graph.nodes[row.name[0]]['y']), (master_graph.nodes[row.name[1]]['x'], master_graph.nodes[row.name[1]]['y'])]), axis=1)

slice_records = []
slice_graphs = []
log_lines = []

for row in admin_units_df.itertuples(index=False):
    municipality_start = time.perf_counter()
    municipality_slug = row.municipality_name_en.lower().replace(' ', '_').replace('-', '_')
    output_path = STEP45_OUTPUT_DIR / f'{municipality_slug}.graphml'
    try:
        polygon_gdf = ox.geocode_to_gdf(row.osm_query_name)
        municipality_polygon = polygon_gdf.geometry.union_all()
        edge_mask = master_edges_gdf.intersects(municipality_polygon)
        clipped_edges_gdf = master_edges_gdf.loc[edge_mask]
        inside_nodes = set(master_nodes_gdf.loc[master_nodes_gdf.intersects(municipality_polygon)].index)
        edge_nodes = set(clipped_edges_gdf.index.get_level_values('u')) | set(clipped_edges_gdf.index.get_level_values('v'))
        municipality_nodes = inside_nodes | edge_nodes
        municipality_graph = master_graph.subgraph(municipality_nodes).copy()
        ox.save_graphml(municipality_graph, output_path)
        file_size_mb = round(output_path.stat().st_size / (1024 ** 2), 2)
        elapsed_seconds = round(time.perf_counter() - municipality_start, 2)
        slice_graphs.append(municipality_graph)
        slice_records.append(
            {
                'municipality_name_en': row.municipality_name_en,
                'osm_query_name': row.osm_query_name,
                'status': 'success',
                'node_count': municipality_graph.number_of_nodes(),
                'edge_count': municipality_graph.number_of_edges(),
                'file_size_mb': file_size_mb,
                'elapsed_seconds': elapsed_seconds,
                'output_path': str(output_path),
                'error_message': '',
            }
        )
        log_lines.append(f"SUCCESS\t{row.municipality_name_en}\t{municipality_graph.number_of_nodes()}\t{municipality_graph.number_of_edges()}\t{file_size_mb}\t{elapsed_seconds}\t{output_path}")
    except Exception as exc:
        elapsed_seconds = round(time.perf_counter() - municipality_start, 2)
        slice_records.append(
            {
                'municipality_name_en': row.municipality_name_en,
                'osm_query_name': row.osm_query_name,
                'status': 'error',
                'node_count': pd.NA,
                'edge_count': pd.NA,
                'file_size_mb': pd.NA,
                'elapsed_seconds': elapsed_seconds,
                'output_path': str(output_path),
                'error_message': str(exc).replace('\n', ' ')[:1000],
            }
        )
        log_lines.append(f"ERROR\t{row.municipality_name_en}\t{elapsed_seconds}\t{str(exc).replace(chr(10), ' ')[:300]}")

slice_metrics_df = pd.DataFrame(slice_records)
slice_metrics_df.to_csv(STEP45_SLICE_METRICS_PATH, index=False, encoding='utf-8-sig')

merged_from_master_graph = nx.MultiDiGraph()
for graph in slice_graphs:
    merged_from_master_graph = nx.compose(merged_from_master_graph, graph)

ox.save_graphml(merged_from_master_graph, STEP45_MERGED_GRAPH_PATH)
merged_from_master_reload = ox.load_graphml(STEP45_MERGED_GRAPH_PATH)
merged_components = list(nx.weakly_connected_components(merged_from_master_reload))

raw_node_ids = set(master_graph.nodes)
raw_edge_ids = set(master_graph.edges(keys=True))
merged_node_ids = set(merged_from_master_reload.nodes)
merged_edge_ids = set(merged_from_master_reload.edges(keys=True))

compare_metrics_df = pd.DataFrame(
    [
        {
            'raw_node_count': master_graph.number_of_nodes(),
            'raw_edge_count': master_graph.number_of_edges(),
            'merged_node_count': merged_from_master_reload.number_of_nodes(),
            'merged_edge_count': merged_from_master_reload.number_of_edges(),
            'delta_nodes_vs_raw': merged_from_master_reload.number_of_nodes() - master_graph.number_of_nodes(),
            'delta_edges_vs_raw': merged_from_master_reload.number_of_edges() - master_graph.number_of_edges(),
            'missing_nodes_vs_raw': len(raw_node_ids - merged_node_ids),
            'extra_nodes_vs_raw': len(merged_node_ids - raw_node_ids),
            'missing_edges_vs_raw': len(raw_edge_ids - merged_edge_ids),
            'extra_edges_vs_raw': len(merged_edge_ids - raw_edge_ids),
            'merged_weak_component_count': len(merged_components),
            'merged_graph_file_size_mb': round(STEP45_MERGED_GRAPH_PATH.stat().st_size / (1024 ** 2), 2),
            'successful_slice_count': int((slice_metrics_df['status'] == 'success').sum()),
            'error_slice_count': int((slice_metrics_df['status'] == 'error').sum()),
        }
    ]
)
compare_metrics_df.to_csv(STEP45_COMPARE_METRICS_PATH, index=False, encoding='utf-8-sig')

step45_route_df = projected_route_check(merged_from_master_reload, STEP4_ROUTE_POINTS)
step45_route_df['graph_type'] = 'master_split_merged'
step45_route_df = step45_route_df[['graph_type', 'route_name', 'status', 'route_length_km', 'path_node_count', 'error_message']]
step45_route_df.to_csv(STEP45_ROUTE_CHECKS_PATH, index=False, encoding='utf-8-sig')

log_lines.append('SUMMARY')
for column, value in compare_metrics_df.iloc[0].items():
    log_lines.append(f"{column}\t{value}")
STEP45_LOG_PATH.write_text('\n'.join(log_lines), encoding='utf-8')

print('Step 4.5 slice metrics summary:')
print(slice_metrics_df[['municipality_name_en', 'status', 'node_count', 'edge_count', 'file_size_mb', 'elapsed_seconds']].to_string(index=False))
print('\nStep 4.5 compare metrics:')
print(compare_metrics_df.to_string(index=False))
print('\nStep 4.5 route checks:')
print(step45_route_df.to_string(index=False))
print(f'Log path: {STEP45_LOG_PATH}')

compare_metrics_df, step45_route_df


Step 4.5 slice metrics summary:
municipality_name_en  status  node_count  edge_count  file_size_mb  elapsed_seconds
            Yokohama success       80551      220655         91.43            26.51
            Kawasaki success       19511       53545         23.44            17.86
          Sagamihara success       16555       46762         20.86             9.32
            Yokosuka success       11613       30494         12.53            25.14
           Hiratsuka success        7805       22003          8.79             2.80
            Kamakura success        4678       12526          5.49             1.85
            Fujisawa success       11838       32775         13.31             4.25
             Odawara success        6394       16977          7.64             2.29
           Chigasaki success        6437       16767          7.08             1.82
               Zushi success        2051        5186          2.25             0.73
               Miura success        1846    

(   raw_node_count  raw_edge_count  merged_node_count  merged_edge_count  \
 0          223850          608205             223850             608205   
 
    delta_nodes_vs_raw  delta_edges_vs_raw  missing_nodes_vs_raw  \
 0                   0                   0                     0   
 
    extra_nodes_vs_raw  missing_edges_vs_raw  extra_edges_vs_raw  \
 0                   0                     0                   0   
 
    merged_weak_component_count  merged_graph_file_size_mb  \
 0                            1                     259.52   
 
    successful_slice_count  error_slice_count  
 0                      33                  0  ,
             graph_type                                    route_name   status  \
 0  master_split_merged  Kawasaki Station to Great Buddha of Kamakura  success   
 1  master_split_merged           Yokohama Station to Odawara Station  success   
 2  master_split_merged     Fujisawa Station to Hakone-Yumoto Station  success   
 
    route_length_

# Step 5 経路探索検証

神奈川県の代表地点候補 CSV から `point_id` が `001` の地点のみを抽出し、総当たりの OD を作成して経路探索を検証する。対象グラフは一括取得版と、親グラフから切り出して再結合した版の 2 種類とし、距離・所要時間・ノード数・差分を確認する。

## 実行内容

1. `data/raw/route_validation/kanagawa_od_point_candidates.csv` から `001` 地点のみを読み込む
2. 同一点を除く総当たりの OD ペアを作成する
3. 一括取得版と親グラフ分割版の両方で最短経路を探索する
4. `length` と `travel_time` を記録する
5. 両方式の差分を OD ごとに集計する

## 想定入出力

- 入力 1: `data/raw/route_validation/kanagawa_od_point_candidates.csv`
- 入力 2: `data/raw/road_network/kanagawa_drive.graphml`
- 入力 3: `data/processed/kanagawa_municipality_merged_from_master.graphml`
- 出力 1: `outputs/metrics/kanagawa_step5_od_pairs.csv`
- 出力 2: `outputs/metrics/kanagawa_step5_route_results.csv`
- 出力 3: `outputs/metrics/kanagawa_step5_route_differences.csv`
- 出力 4: `logs/001_osm_network_feasibility_validation_plan_step5_<timestamp>.log`

## 出力想定内容

- `001` 地点のみで構成した OD 一覧
- 一括取得版と親グラフ分割版の経路距離、所要時間、ノード数、成否
- 両方式の距離差、所要時間差、ノード数差

## 補足

- 今回の総当たりは `point_id` が `001` の 6 地点を対象とするため、同一点除外後の OD は 30 件となる
- `Step 4.5` が成立していれば、差分は原則 `0` に近いことが期待される


In [21]:
STEP5_POINT_CANDIDATES_PATH = PROJECT_ROOT / 'data' / 'raw' / 'route_validation' / 'kanagawa_od_point_candidates.csv'
STEP5_OD_PAIRS_PATH = OUTPUT_METRICS_DIR / 'kanagawa_step5_od_pairs.csv'
STEP5_RESULTS_PATH = OUTPUT_METRICS_DIR / 'kanagawa_step5_route_results.csv'
STEP5_DIFF_PATH = OUTPUT_METRICS_DIR / 'kanagawa_step5_route_differences.csv'
STEP5_LOG_PATH = LOG_DIR / f"001_osm_network_feasibility_validation_plan_step5_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"


def ensure_length_and_travel_time(graph):
    sample_edge = next(iter(graph.edges(keys=True, data=True)), None)
    if sample_edge is None:
        return graph
    edge_data = sample_edge[3]
    if 'travel_time' not in edge_data:
        graph = ox.add_edge_speeds(graph)
        graph = ox.add_edge_travel_times(graph)
    return graph


def route_check_with_metrics(graph, route_points_df, graph_type):
    graph = ensure_length_and_travel_time(graph.copy())
    projected_graph = ox.project_graph(graph)
    projected_crs = projected_graph.graph['crs']
    projected_crs_value = projected_crs.to_string() if hasattr(projected_crs, 'to_string') else str(projected_crs)
    transformer = Transformer.from_crs('EPSG:4326', projected_crs_value, always_xy=True)
    records = []
    for row in route_points_df.itertuples(index=False):
        try:
            start_x, start_y = transformer.transform(row.from_longitude, row.from_latitude)
            end_x, end_y = transformer.transform(row.to_longitude, row.to_latitude)
            start_node = nearest_node_without_optional_deps(projected_graph, start_x, start_y)
            end_node = nearest_node_without_optional_deps(projected_graph, end_x, end_y)
            route_nodes = nx.shortest_path(graph, start_node, end_node, weight='length')
            route_length_m = nx.path_weight(graph, route_nodes, weight='length')
            route_travel_time_s = nx.path_weight(graph, route_nodes, weight='travel_time')
            records.append(
                {
                    'graph_type': graph_type,
                    'od_id': row.od_id,
                    'from_point_id': row.from_point_id,
                    'from_point_name': row.from_point_name,
                    'from_municipality_name_en': row.from_municipality_name_en,
                    'to_point_id': row.to_point_id,
                    'to_point_name': row.to_point_name,
                    'to_municipality_name_en': row.to_municipality_name_en,
                    'status': 'success',
                    'start_node': start_node,
                    'end_node': end_node,
                    'path_node_count': len(route_nodes),
                    'route_length_km': round(route_length_m / 1000, 2),
                    'travel_time_min': round(route_travel_time_s / 60, 2),
                    'error_message': '',
                }
            )
        except Exception as exc:
            records.append(
                {
                    'graph_type': graph_type,
                    'od_id': row.od_id,
                    'from_point_id': row.from_point_id,
                    'from_point_name': row.from_point_name,
                    'from_municipality_name_en': row.from_municipality_name_en,
                    'to_point_id': row.to_point_id,
                    'to_point_name': row.to_point_name,
                    'to_municipality_name_en': row.to_municipality_name_en,
                    'status': 'error',
                    'start_node': pd.NA,
                    'end_node': pd.NA,
                    'path_node_count': pd.NA,
                    'route_length_km': pd.NA,
                    'travel_time_min': pd.NA,
                    'error_message': str(exc).replace('\n', ' ')[:1000],
                }
            )
    return pd.DataFrame(records)


point_candidates_df = pd.read_csv(STEP5_POINT_CANDIDATES_PATH)
point_candidates_df = point_candidates_df[point_candidates_df['point_id'].str.endswith('001')].copy()
point_candidates_df = point_candidates_df.sort_values(['municipality_name_en', 'point_id']).reset_index(drop=True)

od_records = []
for from_row in point_candidates_df.itertuples(index=False):
    for to_row in point_candidates_df.itertuples(index=False):
        if from_row.point_id == to_row.point_id:
            continue
        od_records.append(
            {
                'od_id': f"{from_row.point_id}__{to_row.point_id}",
                'from_point_id': from_row.point_id,
                'from_point_name': from_row.point_name,
                'from_municipality_name_en': from_row.municipality_name_en,
                'from_latitude': from_row.latitude,
                'from_longitude': from_row.longitude,
                'to_point_id': to_row.point_id,
                'to_point_name': to_row.point_name,
                'to_municipality_name_en': to_row.municipality_name_en,
                'to_latitude': to_row.latitude,
                'to_longitude': to_row.longitude,
            }
        )

step5_od_df = pd.DataFrame(od_records)
step5_od_df.to_csv(STEP5_OD_PAIRS_PATH, index=False, encoding='utf-8-sig')

batch_graph_step5 = ox.load_graphml(GRAPH_PATH)
master_split_graph_step5 = ox.load_graphml(STEP45_MERGED_GRAPH_PATH)

step5_batch_df = route_check_with_metrics(batch_graph_step5, step5_od_df, 'batch')
step5_master_split_df = route_check_with_metrics(master_split_graph_step5, step5_od_df, 'master_split_merged')
step5_results_df = pd.concat([step5_batch_df, step5_master_split_df], ignore_index=True)
step5_results_df.to_csv(STEP5_RESULTS_PATH, index=False, encoding='utf-8-sig')

batch_compare_df = step5_batch_df.add_prefix('batch_')
master_compare_df = step5_master_split_df.add_prefix('master_split_')
step5_diff_df = batch_compare_df.merge(master_compare_df, left_on='batch_od_id', right_on='master_split_od_id', how='inner')
step5_diff_df['od_id'] = step5_diff_df['batch_od_id']
step5_diff_df['route_length_km_delta'] = step5_diff_df['master_split_route_length_km'] - step5_diff_df['batch_route_length_km']
step5_diff_df['travel_time_min_delta'] = step5_diff_df['master_split_travel_time_min'] - step5_diff_df['batch_travel_time_min']
step5_diff_df['path_node_count_delta'] = step5_diff_df['master_split_path_node_count'] - step5_diff_df['batch_path_node_count']
step5_diff_df['status_match'] = step5_diff_df['batch_status'] == step5_diff_df['master_split_status']
step5_diff_df = step5_diff_df[
    [
        'od_id',
        'batch_from_point_name',
        'batch_from_municipality_name_en',
        'batch_to_point_name',
        'batch_to_municipality_name_en',
        'batch_status',
        'master_split_status',
        'status_match',
        'batch_route_length_km',
        'master_split_route_length_km',
        'route_length_km_delta',
        'batch_travel_time_min',
        'master_split_travel_time_min',
        'travel_time_min_delta',
        'batch_path_node_count',
        'master_split_path_node_count',
        'path_node_count_delta',
        'batch_error_message',
        'master_split_error_message',
    ]
]
step5_diff_df.to_csv(STEP5_DIFF_PATH, index=False, encoding='utf-8-sig')

step5_log_lines = [
    'Step 5 route validation started',
    f'Created at: {datetime.now().isoformat()}',
    f'Point candidates path: {STEP5_POINT_CANDIDATES_PATH}',
    f'OD pairs path: {STEP5_OD_PAIRS_PATH}',
    f'Results path: {STEP5_RESULTS_PATH}',
    f'Diff path: {STEP5_DIFF_PATH}',
    f'OD count: {len(step5_od_df)}',
    f'Batch success count: {int((step5_batch_df["status"] == "success").sum())}',
    f'Master split success count: {int((step5_master_split_df["status"] == "success").sum())}',
]
STEP5_LOG_PATH.write_text('\n'.join(step5_log_lines), encoding='utf-8')

print(f'Point candidates path: {STEP5_POINT_CANDIDATES_PATH}')
print(f'OD pairs path: {STEP5_OD_PAIRS_PATH}')
print(f'Results path: {STEP5_RESULTS_PATH}')
print(f'Diff path: {STEP5_DIFF_PATH}')
print(f'Log path: {STEP5_LOG_PATH}')
print(f'OD count: {len(step5_od_df)}')
print('\nOD pairs preview:')
print(step5_od_df[['od_id', 'from_point_name', 'from_municipality_name_en', 'to_point_name', 'to_municipality_name_en']].head(10).to_string(index=False))
print('\nRoute results summary:')
print(step5_results_df.groupby(['graph_type', 'status']).size().reset_index(name='count').to_string(index=False))
print('\nRoute differences preview:')
print(step5_diff_df.head(10).to_string(index=False))

step5_diff_df


Point candidates path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\raw\route_validation\kanagawa_od_point_candidates.csv
OD pairs path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_step5_od_pairs.csv
Results path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_step5_route_results.csv
Diff path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_step5_route_differences.csv
Log path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\logs\001_osm_network_feasibility_validation_plan_step5_20260319_161201.log
OD count: 30

OD pairs preview:
                                       od_id       from_point_name from_municipality_name_en            to_point_name to_municipality_name_en
  k

,od_id,batch_from_point_name,batch_from_municipality_name_en,batch_to_point_name,batch_to_municipality_name_en,batch_status,master_split_status,status_match,batch_route_length_km,master_split_route_length_km,route_length_km_delta,batch_travel_time_min,master_split_travel_time_min,travel_time_min_delta,batch_path_node_count,master_split_path_node_count,path_node_count_delta,batch_error_message,master_split_error_message
0,kanagawa_fujisawa_001__kanagawa_hakone_001,Fujisawa Station,Fujisawa,Hakone-Yumoto Station,Hakone,success,success,True,38.88,38.88,0.0,47.14,47.14,0.0,250,250,0,,
1,kanagawa_fujisawa_001__kanagawa_kamakura_001,Fujisawa Station,Fujisawa,Great Buddha of Kamakura,Kamakura,success,success,True,5.83,5.83,0.0,8.88,8.88,0.0,61,61,0,,
2,kanagawa_fujisawa_001__kanagawa_kawasaki_001,Fujisawa Station,Fujisawa,Kawasaki Station,Kawasaki,success,success,True,31.38,31.38,0.0,39.44,39.44,0.0,273,273,0,,
3,kanagawa_fujisawa_001__kanagawa_odawara_001,Fujisawa Station,Fujisawa,Odawara Station,Odawara,success,success,True,32.69,32.69,0.0,37.34,37.34,0.0,184,184,0,,
4,kanagawa_fujisawa_001__kanagawa_yokohama_001,Fujisawa Station,Fujisawa,Yokohama Station,Yokohama,success,success,True,21.26,21.26,0.0,31.77,31.77,0.0,249,249,0,,
5,kanagawa_hakone_001__kanagawa_fujisawa_001,Hakone-Yumoto Station,Hakone,Fujisawa Station,Fujisawa,success,success,True,38.96,38.96,0.0,46.69,46.69,0.0,267,267,0,,
6,kanagawa_hakone_001__kanagawa_kamakura_001,Hakone-Yumoto Station,Hakone,Great Buddha of Kamakura,Kamakura,success,success,True,44.09,44.09,0.0,54.44,54.44,0.0,327,327,0,,
7,kanagawa_hakone_001__kanagawa_kawasaki_001,Hakone-Yumoto Station,Hakone,Kawasaki Station,Kawasaki,success,success,True,68.95,68.95,0.0,82.65,82.65,0.0,472,472,0,,
8,kanagawa_hakone_001__kanagawa_odawara_001,Hakone-Yumoto Station,Hakone,Odawara Station,Odawara,success,success,True,6.74,6.74,0.0,10.88,10.88,0.0,61,61,0,,
9,kanagawa_hakone_001__kanagawa_yokohama_001,Hakone-Yumoto Station,Hakone,Yokohama Station,Yokohama,success,success,True,58.82,58.82,0.0,74.98,74.98,0.0,448,448,0,,


# Step 5 追加検証: 全地点総当たり

地点候補 CSV に登録した全地点を対象に、同一点を除く総当たりの OD を作成して追加検証する。対象グラフは一括取得版と、親グラフから切り出して再結合した版の 2 種類とし、距離・所要時間・ノード数・差分を確認する。

## 実行内容

1. `data/raw/route_validation/kanagawa_od_point_candidates.csv` から全地点を読み込む
2. 同一点を除く総当たりの OD ペアを作成する
3. 一括取得版と親グラフ分割版の両方で最短経路を探索する
4. `length` と `travel_time` を記録する
5. 両方式の差分を OD ごとに集計する

## 想定入出力

- 入力 1: `data/raw/route_validation/kanagawa_od_point_candidates.csv`
- 入力 2: `data/raw/road_network/kanagawa_drive.graphml`
- 入力 3: `data/processed/kanagawa_municipality_merged_from_master.graphml`
- 出力 1: `outputs/metrics/kanagawa_step5_all_od_pairs.csv`
- 出力 2: `outputs/metrics/kanagawa_step5_all_route_results.csv`
- 出力 3: `outputs/metrics/kanagawa_step5_all_route_differences.csv`
- 出力 4: `logs/001_osm_network_feasibility_validation_plan_step5_all_<timestamp>.log`

## 出力想定内容

- 全地点で構成した OD 一覧
- 一括取得版と親グラフ分割版の経路距離、所要時間、ノード数、成否
- 両方式の距離差、所要時間差、ノード数差

## 補足

- 今回の総当たりは CSV に登録した 12 地点を対象とするため、同一点除外後の OD は 132 件となる
- `Step 4.5` が成立していれば、差分は原則 `0` に近いことが期待される


In [22]:
STEP5_ALL_POINT_CANDIDATES_PATH = PROJECT_ROOT / 'data' / 'raw' / 'route_validation' / 'kanagawa_od_point_candidates.csv'
STEP5_ALL_OD_PAIRS_PATH = OUTPUT_METRICS_DIR / 'kanagawa_step5_all_od_pairs.csv'
STEP5_ALL_RESULTS_PATH = OUTPUT_METRICS_DIR / 'kanagawa_step5_all_route_results.csv'
STEP5_ALL_DIFF_PATH = OUTPUT_METRICS_DIR / 'kanagawa_step5_all_route_differences.csv'
STEP5_ALL_LOG_PATH = LOG_DIR / f"001_osm_network_feasibility_validation_plan_step5_all_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

all_point_candidates_df = pd.read_csv(STEP5_ALL_POINT_CANDIDATES_PATH)
all_point_candidates_df = all_point_candidates_df.sort_values(['municipality_name_en', 'point_id']).reset_index(drop=True)

all_od_records = []
for from_row in all_point_candidates_df.itertuples(index=False):
    for to_row in all_point_candidates_df.itertuples(index=False):
        if from_row.point_id == to_row.point_id:
            continue
        all_od_records.append(
            {
                'od_id': f"{from_row.point_id}__{to_row.point_id}",
                'from_point_id': from_row.point_id,
                'from_point_name': from_row.point_name,
                'from_municipality_name_en': from_row.municipality_name_en,
                'from_latitude': from_row.latitude,
                'from_longitude': from_row.longitude,
                'to_point_id': to_row.point_id,
                'to_point_name': to_row.point_name,
                'to_municipality_name_en': to_row.municipality_name_en,
                'to_latitude': to_row.latitude,
                'to_longitude': to_row.longitude,
            }
        )

step5_all_od_df = pd.DataFrame(all_od_records)
step5_all_od_df.to_csv(STEP5_ALL_OD_PAIRS_PATH, index=False, encoding='utf-8-sig')

batch_graph_step5_all = ox.load_graphml(GRAPH_PATH)
master_split_graph_step5_all = ox.load_graphml(STEP45_MERGED_GRAPH_PATH)

step5_all_batch_df = route_check_with_metrics(batch_graph_step5_all, step5_all_od_df, 'batch')
step5_all_master_split_df = route_check_with_metrics(master_split_graph_step5_all, step5_all_od_df, 'master_split_merged')
step5_all_results_df = pd.concat([step5_all_batch_df, step5_all_master_split_df], ignore_index=True)
step5_all_results_df.to_csv(STEP5_ALL_RESULTS_PATH, index=False, encoding='utf-8-sig')

all_batch_compare_df = step5_all_batch_df.add_prefix('batch_')
all_master_compare_df = step5_all_master_split_df.add_prefix('master_split_')
step5_all_diff_df = all_batch_compare_df.merge(all_master_compare_df, left_on='batch_od_id', right_on='master_split_od_id', how='inner')
step5_all_diff_df['od_id'] = step5_all_diff_df['batch_od_id']
step5_all_diff_df['route_length_km_delta'] = step5_all_diff_df['master_split_route_length_km'] - step5_all_diff_df['batch_route_length_km']
step5_all_diff_df['travel_time_min_delta'] = step5_all_diff_df['master_split_travel_time_min'] - step5_all_diff_df['batch_travel_time_min']
step5_all_diff_df['path_node_count_delta'] = step5_all_diff_df['master_split_path_node_count'] - step5_all_diff_df['batch_path_node_count']
step5_all_diff_df['status_match'] = step5_all_diff_df['batch_status'] == step5_all_diff_df['master_split_status']
step5_all_diff_df = step5_all_diff_df[
    [
        'od_id',
        'batch_from_point_name',
        'batch_from_municipality_name_en',
        'batch_to_point_name',
        'batch_to_municipality_name_en',
        'batch_status',
        'master_split_status',
        'status_match',
        'batch_route_length_km',
        'master_split_route_length_km',
        'route_length_km_delta',
        'batch_travel_time_min',
        'master_split_travel_time_min',
        'travel_time_min_delta',
        'batch_path_node_count',
        'master_split_path_node_count',
        'path_node_count_delta',
        'batch_error_message',
        'master_split_error_message',
    ]
]
step5_all_diff_df.to_csv(STEP5_ALL_DIFF_PATH, index=False, encoding='utf-8-sig')

step5_all_log_lines = [
    'Step 5 all-points route validation started',
    f'Created at: {datetime.now().isoformat()}',
    f'Point candidates path: {STEP5_ALL_POINT_CANDIDATES_PATH}',
    f'OD pairs path: {STEP5_ALL_OD_PAIRS_PATH}',
    f'Results path: {STEP5_ALL_RESULTS_PATH}',
    f'Diff path: {STEP5_ALL_DIFF_PATH}',
    f'OD count: {len(step5_all_od_df)}',
    f'Batch success count: {int((step5_all_batch_df["status"] == "success").sum())}',
    f'Master split success count: {int((step5_all_master_split_df["status"] == "success").sum())}',
]
STEP5_ALL_LOG_PATH.write_text('\n'.join(step5_all_log_lines), encoding='utf-8')

print(f'Point candidates path: {STEP5_ALL_POINT_CANDIDATES_PATH}')
print(f'OD pairs path: {STEP5_ALL_OD_PAIRS_PATH}')
print(f'Results path: {STEP5_ALL_RESULTS_PATH}')
print(f'Diff path: {STEP5_ALL_DIFF_PATH}')
print(f'Log path: {STEP5_ALL_LOG_PATH}')
print(f'OD count: {len(step5_all_od_df)}')
print('\nOD pairs preview:')
print(step5_all_od_df[['od_id', 'from_point_name', 'from_municipality_name_en', 'to_point_name', 'to_municipality_name_en']].head(10).to_string(index=False))
print('\nRoute results summary:')
print(step5_all_results_df.groupby(['graph_type', 'status']).size().reset_index(name='count').to_string(index=False))
print('\nRoute differences preview:')
print(step5_all_diff_df.head(10).to_string(index=False))

step5_all_diff_df


Point candidates path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\raw\route_validation\kanagawa_od_point_candidates.csv
OD pairs path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_step5_all_od_pairs.csv
Results path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_step5_all_route_results.csv
Diff path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_step5_all_route_differences.csv
Log path: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\logs\001_osm_network_feasibility_validation_plan_step5_all_20260319_162549.log
OD count: 132

OD pairs preview:
                                       od_id  from_point_name from_municipality_name_en            to_point_name to_municipality

,od_id,batch_from_point_name,batch_from_municipality_name_en,batch_to_point_name,batch_to_municipality_name_en,batch_status,master_split_status,status_match,batch_route_length_km,master_split_route_length_km,route_length_km_delta,batch_travel_time_min,master_split_travel_time_min,travel_time_min_delta,batch_path_node_count,master_split_path_node_count,path_node_count_delta,batch_error_message,master_split_error_message
0,kanagawa_fujisawa_001__kanagawa_fujisawa_002,Fujisawa Station,Fujisawa,Enoshima,Fujisawa,success,success,True,5.22,5.22,0.0,8.38,8.38,0.0,57,57,0,,
1,kanagawa_fujisawa_001__kanagawa_hakone_001,Fujisawa Station,Fujisawa,Hakone-Yumoto Station,Hakone,success,success,True,38.88,38.88,0.0,47.14,47.14,0.0,250,250,0,,
2,kanagawa_fujisawa_001__kanagawa_hakone_002,Fujisawa Station,Fujisawa,Owakudani,Hakone,success,success,True,52.46,52.46,0.0,67.78,67.78,0.0,289,289,0,,
3,kanagawa_fujisawa_001__kanagawa_kamakura_001,Fujisawa Station,Fujisawa,Great Buddha of Kamakura,Kamakura,success,success,True,5.83,5.83,0.0,8.88,8.88,0.0,61,61,0,,
4,kanagawa_fujisawa_001__kanagawa_kamakura_002,Fujisawa Station,Fujisawa,Tsurugaoka Hachimangu,Kamakura,success,success,True,8.00,8.00,0.0,12.68,12.68,0.0,80,80,0,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,kanagawa_yokohama_002__kanagawa_kawasaki_001,Yokohama Red Brick Warehouse,Yokohama,Kawasaki Station,Kawasaki,success,success,True,12.56,12.56,0.0,18.39,18.39,0.0,109,109,0,,
128,kanagawa_yokohama_002__kanagawa_kawasaki_002,Yokohama Red Brick Warehouse,Yokohama,Kawasaki Daishi,Kawasaki,success,success,True,14.45,14.45,0.0,20.01,20.01,0.0,142,142,0,,
129,kanagawa_yokohama_002__kanagawa_odawara_001,Yokohama Red Brick Warehouse,Yokohama,Odawara Station,Odawara,success,success,True,53.44,53.44,0.0,67.73,67.73,0.0,386,386,0,,
130,kanagawa_yokohama_002__kanagawa_odawara_002,Yokohama Red Brick Warehouse,Yokohama,Odawara Castle,Odawara,success,success,True,53.87,53.87,0.0,68.45,68.45,0.0,397,397,0,,


# Step 6 比較準備: 緯度経度付き比較ファイル作成

Google Maps 側で距離・所要時間を取得しやすいように、OSMnx の経路探索結果へ From / To の緯度経度を付与した比較準備ファイルを作成する。あわせて、Google Maps 側の値を追記するためのテンプレート CSV も出力する。

## 実行内容

1. Step 5 の OD 一覧と経路探索結果を読み込む
2. OSMnx 結果へ From / To の緯度経度を結合する
3. Google Maps 側の距離・所要時間を追記するための空列を持つテンプレート CSV を作成する
4. 30 OD 版と全地点版の両方を出力する

## 想定入出力

- 入力: `outputs/metrics/kanagawa_step5_*.csv`
- 出力 1: `outputs/metrics/kanagawa_step5_route_results_with_coords.csv`
- 出力 2: `outputs/metrics/kanagawa_step5_all_route_results_with_coords.csv`
- 出力 3: `data/raw/route_validation/kanagawa_step6_google_maps_30od_template.csv`
- 出力 4: `data/raw/route_validation/kanagawa_step6_google_maps_all_template.csv`

## 補足

- 実際に Google Maps API や GAS で取得した CSV は `data/raw/route_validation/` に配置する前提とする
- 差分集計は、Google Maps 側の列を埋めた CSV を戻した後に実施する


In [23]:
STEP6_PREP_30_RESULTS_WITH_COORDS = OUTPUT_METRICS_DIR / 'kanagawa_step5_route_results_with_coords.csv'
STEP6_PREP_ALL_RESULTS_WITH_COORDS = OUTPUT_METRICS_DIR / 'kanagawa_step5_all_route_results_with_coords.csv'
STEP6_GOOGLE_30_TEMPLATE = PROJECT_ROOT / 'data' / 'raw' / 'route_validation' / 'kanagawa_step6_google_maps_30od_template.csv'
STEP6_GOOGLE_ALL_TEMPLATE = PROJECT_ROOT / 'data' / 'raw' / 'route_validation' / 'kanagawa_step6_google_maps_all_template.csv'


def build_compare_prep_files(od_pairs_path, route_results_path, output_with_coords_path, google_template_path):
    od_df = pd.read_csv(od_pairs_path)
    route_df = pd.read_csv(route_results_path)

    merged_df = route_df.merge(
        od_df,
        on=[
            'od_id',
            'from_point_id',
            'from_point_name',
            'from_municipality_name_en',
            'to_point_id',
            'to_point_name',
            'to_municipality_name_en',
        ],
        how='left',
    )

    result_columns = [
        'graph_type',
        'od_id',
        'from_point_id',
        'from_point_name',
        'from_municipality_name_en',
        'from_latitude',
        'from_longitude',
        'to_point_id',
        'to_point_name',
        'to_municipality_name_en',
        'to_latitude',
        'to_longitude',
        'status',
        'start_node',
        'end_node',
        'path_node_count',
        'route_length_km',
        'travel_time_min',
        'error_message',
    ]
    merged_df = merged_df[result_columns]
    merged_df.to_csv(output_with_coords_path, index=False, encoding='utf-8-sig')

    google_template_df = merged_df[merged_df['graph_type'] == 'batch'].copy()
    google_template_df = google_template_df[
        [
            'od_id',
            'from_point_id',
            'from_point_name',
            'from_municipality_name_en',
            'from_latitude',
            'from_longitude',
            'to_point_id',
            'to_point_name',
            'to_municipality_name_en',
            'to_latitude',
            'to_longitude',
            'route_length_km',
            'travel_time_min',
        ]
    ].copy()
    google_template_df = google_template_df.rename(
        columns={
            'route_length_km': 'osmnx_route_length_km',
            'travel_time_min': 'osmnx_travel_time_min',
        }
    )
    google_template_df['google_maps_distance_km'] = pd.NA
    google_template_df['google_maps_time_min'] = pd.NA
    google_template_df['google_maps_checked_at'] = pd.NA
    google_template_df['google_maps_notes'] = pd.NA
    google_template_df.to_csv(google_template_path, index=False, encoding='utf-8-sig')

    return merged_df, google_template_df


step6_prep_30_df, step6_google_30_template_df = build_compare_prep_files(
    STEP5_OD_PAIRS_PATH,
    STEP5_RESULTS_PATH,
    STEP6_PREP_30_RESULTS_WITH_COORDS,
    STEP6_GOOGLE_30_TEMPLATE,
)

step6_prep_all_df, step6_google_all_template_df = build_compare_prep_files(
    STEP5_ALL_OD_PAIRS_PATH,
    STEP5_ALL_RESULTS_PATH,
    STEP6_PREP_ALL_RESULTS_WITH_COORDS,
    STEP6_GOOGLE_ALL_TEMPLATE,
)

print(f'30 OD results with coords: {STEP6_PREP_30_RESULTS_WITH_COORDS}')
print(f'All OD results with coords: {STEP6_PREP_ALL_RESULTS_WITH_COORDS}')
print(f'30 OD Google template: {STEP6_GOOGLE_30_TEMPLATE}')
print(f'All OD Google template: {STEP6_GOOGLE_ALL_TEMPLATE}')
print('\n30 OD Google template preview:')
print(step6_google_30_template_df.head(10).to_string(index=False))

step6_google_30_template_df


30 OD results with coords: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_step5_route_results_with_coords.csv
All OD results with coords: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\outputs\metrics\kanagawa_step5_all_route_results_with_coords.csv
30 OD Google template: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\raw\route_validation\kanagawa_step6_google_maps_30od_template.csv
All OD Google template: C:\Users\LCCadmin\OneDrive - 株式会社　ロジクロス・コミュニケーション\ドキュメント\GitHub\vrp-road-network-optimization\data\raw\route_validation\kanagawa_step6_google_maps_all_template.csv

30 OD Google template preview:
                                       od_id         from_point_id       from_point_name from_municipality_name_en  from_latitude  from_longitude           to_point_id            to_point_name to_municipality_name_en  to_la

,od_id,from_point_id,from_point_name,from_municipality_name_en,from_latitude,from_longitude,to_point_id,to_point_name,to_municipality_name_en,to_latitude,to_longitude,osmnx_route_length_km,osmnx_travel_time_min,google_maps_distance_km,google_maps_time_min,google_maps_checked_at,google_maps_notes
0,kanagawa_fujisawa_001__kanagawa_hakone_001,kanagawa_fujisawa_001,Fujisawa Station,Fujisawa,35.3388,139.4876,kanagawa_hakone_001,Hakone-Yumoto Station,Hakone,35.2323,139.1069,38.88,47.14,<NA>,<NA>,<NA>,<NA>
1,kanagawa_fujisawa_001__kanagawa_kamakura_001,kanagawa_fujisawa_001,Fujisawa Station,Fujisawa,35.3388,139.4876,kanagawa_kamakura_001,Great Buddha of Kamakura,Kamakura,35.3167,139.5354,5.83,8.88,<NA>,<NA>,<NA>,<NA>
2,kanagawa_fujisawa_001__kanagawa_kawasaki_001,kanagawa_fujisawa_001,Fujisawa Station,Fujisawa,35.3388,139.4876,kanagawa_kawasaki_001,Kawasaki Station,Kawasaki,35.5314,139.6969,31.38,39.44,<NA>,<NA>,<NA>,<NA>
3,kanagawa_fujisawa_001__kanagawa_odawara_001,kanagawa_fujisawa_001,Fujisawa Station,Fujisawa,35.3388,139.4876,kanagawa_odawara_001,Odawara Station,Odawara,35.2556,139.1597,32.69,37.34,<NA>,<NA>,<NA>,<NA>
4,kanagawa_fujisawa_001__kanagawa_yokohama_001,kanagawa_fujisawa_001,Fujisawa Station,Fujisawa,35.3388,139.4876,kanagawa_yokohama_001,Yokohama Station,Yokohama,35.4662,139.6227,21.26,31.77,<NA>,<NA>,<NA>,<NA>
5,kanagawa_hakone_001__kanagawa_fujisawa_001,kanagawa_hakone_001,Hakone-Yumoto Station,Hakone,35.2323,139.1069,kanagawa_fujisawa_001,Fujisawa Station,Fujisawa,35.3388,139.4876,38.96,46.69,<NA>,<NA>,<NA>,<NA>
6,kanagawa_hakone_001__kanagawa_kamakura_001,kanagawa_hakone_001,Hakone-Yumoto Station,Hakone,35.2323,139.1069,kanagawa_kamakura_001,Great Buddha of Kamakura,Kamakura,35.3167,139.5354,44.09,54.44,<NA>,<NA>,<NA>,<NA>
7,kanagawa_hakone_001__kanagawa_kawasaki_001,kanagawa_hakone_001,Hakone-Yumoto Station,Hakone,35.2323,139.1069,kanagawa_kawasaki_001,Kawasaki Station,Kawasaki,35.5314,139.6969,68.95,82.65,<NA>,<NA>,<NA>,<NA>
8,kanagawa_hakone_001__kanagawa_odawara_001,kanagawa_hakone_001,Hakone-Yumoto Station,Hakone,35.2323,139.1069,kanagawa_odawara_001,Odawara Station,Odawara,35.2556,139.1597,6.74,10.88,<NA>,<NA>,<NA>,<NA>
9,kanagawa_hakone_001__kanagawa_yokohama_001,kanagawa_hakone_001,Hakone-Yumoto Station,Hakone,35.2323,139.1069,kanagawa_yokohama_001,Yokohama Station,Yokohama,35.4662,139.6227,58.82,74.98,<NA>,<NA>,<NA>,<NA>


# Step 6 妥当性確認

OSMnx と Google Maps の比較は、単に差があるかではなく、**その差が実務上許容できるか** を判断するために行う。ここでは、対応のある OD データに対して、記述統計、Wilcoxon 検定、効果量、実務閾値判定、順位一致率を順に確認する。

## この Step で行うこと

1. OSMnx と Google Maps の距離・時間差分を読み込み、比較用データセットを作成する
2. 距離差、時間差、誤差率の記述統計を確認する
3. Wilcoxon 符号付順位検定で差が統計的に一貫しているかを見る
4. 効果量で差の大きさを確認する
5. 実務閾値で PoC として許容できるかを判定する
6. 同一出発点に対する候補順位がどの程度一致するかを確認する

## OSMnx ≒ Google Maps と判断する基準

- 距離: 平均絶対誤差率 `< 10%`、`p95 < 20%`、最大絶対誤差率 `< 30%`
- 時間: 平均絶対誤差率 `< 20%`、`p95 < 30%`、最大絶対誤差率 `< 40%`
- 順位: Top1 一致率 `>= 90%`、平均 Spearman 相関 `>= 0.9`
- 検定で有意差が出ても、効果量が小さく実務閾値を満たす場合は PoC 上は許容とみなす

## 出力先

- `outputs/validation/step6/` に、比較基礎データ、各検証結果、総合判定をファイル単位で出力する
- `STEP6_VARIANT` を `30od` または `all` に変えることで対象を切り替えられる


In [ ]:
from scipy import stats

STEP6_VARIANT = '30od'  # '30od' or 'all'
STEP6_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'validation' / 'step6'
STEP6_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STEP6_INPUTS = {
    '30od': {
        'google_path': PROJECT_ROOT / 'data' / 'raw' / 'route_validation' / 'kanagawa_step6_google_maps_30od.csv',
        'label': 'kanagawa_step6_30od',
    },
    'all': {
        'google_path': PROJECT_ROOT / 'data' / 'raw' / 'route_validation' / 'kanagawa_step6_google_maps_all.csv',
        'label': 'kanagawa_step6_all',
    },
}

if STEP6_VARIANT not in STEP6_INPUTS:
    raise ValueError(f'Unknown STEP6_VARIANT: {STEP6_VARIANT}')

step6_google_path = STEP6_INPUTS[STEP6_VARIANT]['google_path']
step6_label = STEP6_INPUTS[STEP6_VARIANT]['label']
step6_base_path = STEP6_OUTPUT_DIR / f'{step6_label}_base_comparison.csv'
step6_summary_path = STEP6_OUTPUT_DIR / f'{step6_label}_descriptive_stats.csv'
step6_wilcoxon_path = STEP6_OUTPUT_DIR / f'{step6_label}_wilcoxon.csv'
step6_effect_size_path = STEP6_OUTPUT_DIR / f'{step6_label}_effect_size.csv'
step6_threshold_path = STEP6_OUTPUT_DIR / f'{step6_label}_threshold_judgement.csv'
step6_ranking_detail_path = STEP6_OUTPUT_DIR / f'{step6_label}_ranking_detail.csv'
step6_ranking_summary_path = STEP6_OUTPUT_DIR / f'{step6_label}_ranking_summary.csv'

step6_df = pd.read_csv(step6_google_path)
step6_df['google_maps_distance_km'] = pd.to_numeric(step6_df['google_maps_distance_km'], errors='coerce')
step6_df['google_maps_time_min'] = pd.to_numeric(step6_df['google_maps_time_min'], errors='coerce')
step6_df['osmnx_route_length_km'] = pd.to_numeric(step6_df['osmnx_route_length_km'], errors='coerce')
step6_df['osmnx_travel_time_min'] = pd.to_numeric(step6_df['osmnx_travel_time_min'], errors='coerce')
step6_df['diff_distance_km'] = step6_df['osmnx_route_length_km'] - step6_df['google_maps_distance_km']
step6_df['diff_time_min'] = step6_df['osmnx_travel_time_min'] - step6_df['google_maps_time_min']
step6_df['distance_error_rate'] = step6_df['diff_distance_km'] / step6_df['google_maps_distance_km']
step6_df['time_error_rate'] = step6_df['diff_time_min'] / step6_df['google_maps_time_min']
step6_df['abs_distance_error_rate'] = step6_df['distance_error_rate'].abs()
step6_df['abs_time_error_rate'] = step6_df['time_error_rate'].abs()
step6_df.to_csv(step6_base_path, index=False, encoding='utf-8-sig')

print(f'Step 6 variant: {STEP6_VARIANT}')
print(f'Google input path: {step6_google_path}')
print(f'Base comparison path: {step6_base_path}')
print(f'OD count: {len(step6_df)}')
print(step6_df[['od_id', 'osmnx_route_length_km', 'google_maps_distance_km', 'osmnx_travel_time_min', 'google_maps_time_min']].head(10).to_string(index=False))

step6_df.head()


In [ ]:
def descriptive_summary(series):
    clean = series.dropna()
    return {
        'count': int(clean.count()),
        'mean': clean.mean(),
        'median': clean.median(),
        'std': clean.std(),
        'p90': clean.quantile(0.90),
        'p95': clean.quantile(0.95),
        'max': clean.max(),
        'min': clean.min(),
    }


step6_summary_records = []
for metric_name, series in {
    'diff_distance_km': step6_df['diff_distance_km'],
    'diff_time_min': step6_df['diff_time_min'],
    'distance_error_rate': step6_df['distance_error_rate'],
    'time_error_rate': step6_df['time_error_rate'],
    'abs_distance_error_rate': step6_df['abs_distance_error_rate'],
    'abs_time_error_rate': step6_df['abs_time_error_rate'],
}.items():
    record = {'metric': metric_name}
    record.update(descriptive_summary(series))
    step6_summary_records.append(record)

step6_summary_df = pd.DataFrame(step6_summary_records)
step6_summary_df.to_csv(step6_summary_path, index=False, encoding='utf-8-sig')

display_summary_df = step6_summary_df.copy()
rate_mask = display_summary_df['metric'].str.contains('error_rate')
for column in ['mean', 'median', 'std', 'p90', 'p95', 'max', 'min']:
    display_summary_df.loc[rate_mask, column] = display_summary_df.loc[rate_mask, column] * 100

print(f'Descriptive stats path: {step6_summary_path}')
print(display_summary_df.to_string(index=False))

display_summary_df


In [ ]:
def safe_wilcoxon(series):
    clean = series.dropna()
    if clean.empty:
        return {'n': 0, 'statistic': pd.NA, 'p_value': pd.NA, 'note': 'no_data'}
    if (clean == 0).all():
        return {'n': int(clean.count()), 'statistic': 0.0, 'p_value': 1.0, 'note': 'all_zero'}
    result = stats.wilcoxon(clean)
    return {'n': int(clean.count()), 'statistic': result.statistic, 'p_value': result.pvalue, 'note': ''}


step6_wilcoxon_df = pd.DataFrame(
    [
        {'metric': 'diff_distance_km', **safe_wilcoxon(step6_df['diff_distance_km'])},
        {'metric': 'diff_time_min', **safe_wilcoxon(step6_df['diff_time_min'])},
    ]
)
step6_wilcoxon_df['significant_at_5pct'] = step6_wilcoxon_df['p_value'].apply(lambda x: pd.NA if pd.isna(x) else x < 0.05)
step6_wilcoxon_df.to_csv(step6_wilcoxon_path, index=False, encoding='utf-8-sig')

print(f'Wilcoxon path: {step6_wilcoxon_path}')
print(step6_wilcoxon_df.to_string(index=False))

step6_wilcoxon_df


In [ ]:
def paired_cohens_d(series):
    clean = series.dropna()
    if clean.empty:
        return pd.NA
    std = clean.std()
    if std == 0:
        return 0.0 if clean.mean() == 0 else pd.NA
    return clean.mean() / std


def effect_size_label(value):
    if pd.isna(value):
        return 'undefined'
    abs_value = abs(value)
    if abs_value < 0.2:
        return 'negligible'
    if abs_value < 0.5:
        return 'small'
    if abs_value < 0.8:
        return 'medium'
    return 'large'


step6_effect_df = pd.DataFrame(
    [
        {'metric': 'diff_distance_km', 'cohens_d': paired_cohens_d(step6_df['diff_distance_km'])},
        {'metric': 'diff_time_min', 'cohens_d': paired_cohens_d(step6_df['diff_time_min'])},
    ]
)
step6_effect_df['effect_size_label'] = step6_effect_df['cohens_d'].apply(effect_size_label)
step6_effect_df.to_csv(step6_effect_size_path, index=False, encoding='utf-8-sig')

print(f'Effect size path: {step6_effect_size_path}')
print(step6_effect_df.to_string(index=False))

step6_effect_df


In [ ]:
step6_threshold_df = pd.DataFrame(
    [
        {
            'metric': 'distance_abs_error_rate',
            'mean_pct': step6_df['abs_distance_error_rate'].mean() * 100,
            'p95_pct': step6_df['abs_distance_error_rate'].quantile(0.95) * 100,
            'max_pct': step6_df['abs_distance_error_rate'].max() * 100,
            'mean_threshold_pct': 10.0,
            'p95_threshold_pct': 20.0,
            'max_threshold_pct': 30.0,
        },
        {
            'metric': 'time_abs_error_rate',
            'mean_pct': step6_df['abs_time_error_rate'].mean() * 100,
            'p95_pct': step6_df['abs_time_error_rate'].quantile(0.95) * 100,
            'max_pct': step6_df['abs_time_error_rate'].max() * 100,
            'mean_threshold_pct': 20.0,
            'p95_threshold_pct': 30.0,
            'max_threshold_pct': 40.0,
        },
    ]
)
step6_threshold_df['mean_pass'] = step6_threshold_df['mean_pct'] < step6_threshold_df['mean_threshold_pct']
step6_threshold_df['p95_pass'] = step6_threshold_df['p95_pct'] < step6_threshold_df['p95_threshold_pct']
step6_threshold_df['max_pass'] = step6_threshold_df['max_pct'] < step6_threshold_df['max_threshold_pct']
step6_threshold_df['overall_pass'] = step6_threshold_df[['mean_pass', 'p95_pass', 'max_pass']].all(axis=1)
step6_threshold_df.to_csv(step6_threshold_path, index=False, encoding='utf-8-sig')

overall_practical_match = bool(step6_threshold_df['overall_pass'].all())
print(f'Threshold judgement path: {step6_threshold_path}')
print(step6_threshold_df.to_string(index=False))
print(f'Practical threshold overall pass: {overall_practical_match}')

step6_threshold_df


In [ ]:
ranking_records = []
for origin_id, group_df in step6_df.groupby('from_point_id'):
    if len(group_df) < 2:
        continue
    osm_distance_order = group_df.sort_values(['osmnx_route_length_km', 'to_point_id'])['to_point_id'].tolist()
    google_distance_order = group_df.sort_values(['google_maps_distance_km', 'to_point_id'])['to_point_id'].tolist()
    osm_time_order = group_df.sort_values(['osmnx_travel_time_min', 'to_point_id'])['to_point_id'].tolist()
    google_time_order = group_df.sort_values(['google_maps_time_min', 'to_point_id'])['to_point_id'].tolist()

    distance_rank_df = group_df[['to_point_id', 'osmnx_route_length_km', 'google_maps_distance_km']].copy()
    distance_rank_df['osmnx_rank'] = distance_rank_df['osmnx_route_length_km'].rank(method='first')
    distance_rank_df['google_rank'] = distance_rank_df['google_maps_distance_km'].rank(method='first')
    time_rank_df = group_df[['to_point_id', 'osmnx_travel_time_min', 'google_maps_time_min']].copy()
    time_rank_df['osmnx_rank'] = time_rank_df['osmnx_travel_time_min'].rank(method='first')
    time_rank_df['google_rank'] = time_rank_df['google_maps_time_min'].rank(method='first')

    distance_spearman = stats.spearmanr(distance_rank_df['osmnx_rank'], distance_rank_df['google_rank']).statistic
    time_spearman = stats.spearmanr(time_rank_df['osmnx_rank'], time_rank_df['google_rank']).statistic

    ranking_records.append(
        {
            'from_point_id': origin_id,
            'from_point_name': group_df['from_point_name'].iloc[0],
            'from_municipality_name_en': group_df['from_municipality_name_en'].iloc[0],
            'candidate_count': len(group_df),
            'distance_exact_match': osm_distance_order == google_distance_order,
            'distance_top1_match': osm_distance_order[0] == google_distance_order[0],
            'distance_spearman': distance_spearman,
            'time_exact_match': osm_time_order == google_time_order,
            'time_top1_match': osm_time_order[0] == google_time_order[0],
            'time_spearman': time_spearman,
        }
    )

step6_ranking_detail_df = pd.DataFrame(ranking_records)
step6_ranking_detail_df.to_csv(step6_ranking_detail_path, index=False, encoding='utf-8-sig')

step6_ranking_summary_df = pd.DataFrame(
    [
        {
            'distance_exact_match_rate': step6_ranking_detail_df['distance_exact_match'].mean(),
            'distance_top1_match_rate': step6_ranking_detail_df['distance_top1_match'].mean(),
            'distance_mean_spearman': step6_ranking_detail_df['distance_spearman'].mean(),
            'time_exact_match_rate': step6_ranking_detail_df['time_exact_match'].mean(),
            'time_top1_match_rate': step6_ranking_detail_df['time_top1_match'].mean(),
            'time_mean_spearman': step6_ranking_detail_df['time_spearman'].mean(),
            'distance_top1_pass': step6_ranking_detail_df['distance_top1_match'].mean() >= 0.9,
            'distance_spearman_pass': step6_ranking_detail_df['distance_spearman'].mean() >= 0.9,
            'time_top1_pass': step6_ranking_detail_df['time_top1_match'].mean() >= 0.9,
            'time_spearman_pass': step6_ranking_detail_df['time_spearman'].mean() >= 0.9,
        }
    ]
)
step6_ranking_summary_df.to_csv(step6_ranking_summary_path, index=False, encoding='utf-8-sig')

print(f'Ranking detail path: {step6_ranking_detail_path}')
print(f'Ranking summary path: {step6_ranking_summary_path}')
print('\nRanking detail:')
print(step6_ranking_detail_df.to_string(index=False))
print('\nRanking summary:')
print(step6_ranking_summary_df.to_string(index=False))

step6_ranking_summary_df
